<div style="background: linear-gradient(120deg, #0b1f4b 0%, #173d7a 38%, #2a78d6 72%, #eb6834 100%); padding: 32px 34px 28px 34px; border-radius: 14px; color: #ffffff;">
<div style="font-size: 12px; letter-spacing: 3px; text-transform: uppercase; opacity: 0.85;">Playground Series S6E11 &middot; Predicting Credit Card Fraud &middot; ROC-AUC</div>
<div style="font-size: 36px; font-weight: 800; line-height: 1.12; margin-top: 10px;">How much of the public top is real?</div>
<div style="font-size: 17px; margin-top: 8px; opacity: 0.95;">Public ladders, their noise, the winner's curse, and the nested CV that tells you which steps will survive the private 80%</div>
<div style="margin-top: 18px;">
<span style="background: #ffffff; color: #0b1f4b; border-radius: 12px; padding: 3px 11px; font-size: 12px; font-weight: 700; margin-right: 6px;">public LB = 20% of test</span>
<span style="background: #ffffff; color: #0b1f4b; border-radius: 12px; padding: 3px 11px; font-size: 12px; font-weight: 700; margin-right: 6px;">private = 80%</span>
<span style="background: #ffffff; color: #0b1f4b; border-radius: 12px; padding: 3px 11px; font-size: 12px; font-weight: 700; margin-right: 6px;">nested CV</span>
<span style="background: rgba(255,255,255,0.18); color: #ffffff; border-radius: 12px; padding: 3px 11px; font-size: 12px; margin-right: 6px;">every number computed here</span>
<span style="background: rgba(255,255,255,0.18); color: #ffffff; border-radius: 12px; padding: 3px 11px; font-size: 12px; margin-right: 6px;">public inputs only, SHA-256 checked</span>
<span style="background: rgba(255,255,255,0.18); color: #ffffff; border-radius: 12px; padding: 3px 11px; font-size: 12px;">CPU, ~15 min</span>
</div>
</div>

The top of the public board is a dense band of files a few hundred-thousandths apart. Many of them are **ladders**:
a strong public file plus small steps, each kept because the public score went up by one or two points
(1 point = 0.00001 AUC). But the public score is computed on only ~60,000 rows with about a thousand frauds. This
notebook measures, with public files only, **how big the noise of one such step is, how much of a ladder's gain
nested CV confirms, and how much of the top band is likely to move on the private 80%**. It takes no side: the ladder
authors themselves list every probe and check their steps on CV, which is what makes this analysis possible.

<div style="background: #fff4ec; border-left: 5px solid #eb6834; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>New in v4 (14 Nov).</b><br>
&bull; <a href="#s8">The board, 12 &rarr; 14 Nov</a>: the band rose; #1 sits far outside any public-noise range.<br>
&bull; <a href="#s9">The two new top ladders</a> (goodpjw2008's <i>Public and CV Up</i>, 0.95386, and Areeb Ali's <i>Bold, Then Bolder</i>, 0.95388)
rebuilt from public files: every probe, its noise, its OOF change, and its expected private value.<br>
&bull; <a href="#s10">"Public and CV up", checked fold by fold</a>, with a nested choice of the last step size.<br>
&bull; <a href="#s11">Error analysis</a>: which rows the top files fight over, by score band, amount and channel.<br>
&bull; <a href="#s12">An honest extension of the best public file</a>: up to two steps, chosen by nested CV only. Result: no step
passes, so <code>submission.csv</code> is the plain, credited rebuild of the 0.95388 file.
</div>

<div style="background: #eef4fc; border-left: 5px solid #2a78d6; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>Contents</b><br>
<a href="#ack">0. Acknowledgements and who did what</a><br>
<a href="#hero">The whole story in one figure</a><br>
<a href="#s1">1. The competition in five minutes</a> &mdash; data, AUC, how big the public and private parts are<br>
<a href="#s2">2. How noisy is the public LB?</a> &mdash; one file versus the difference of two files, and a one-line rule<br>
<a href="#s3">3. Anatomy of a ladder</a> &mdash; every public rung against its noise and against nested CV<br>
<a href="#s4">4. The winner's curse</a> &mdash; why the best of many probes looks better than it is<br>
<a href="#s5">5. CV can be tuned too</a> &mdash; early stopping on the scored fold, selection on the reported folds, tests inside one fold<br>
<a href="#s6">6. What did not work</a><br>
<a href="#s7">7. What to expect on private, and how to pick two final files</a><br>
<a href="#s8">8. New: the board, 12 &rarr; 14 November</a><br>
<a href="#s9">9. New: anatomy of the two new ladders</a> &mdash; and what each new notebook's public tuning is likely to cost on private<br>
<a href="#s10">10. New: "public and CV up", checked fold by fold</a><br>
<a href="#s11">11. New: error analysis on the OOF rows</a><br>
<a href="#s12">12. New: one honest step on top of the best public file?</a> &mdash; this notebook's <code>submission.csv</code><br>
<a href="#s13">13. Full credits and licences</a>
</div>

If this is useful to you, an upvote helps others find it. Corrections are very welcome in the comments.

<a id="ack"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">0 &middot; Acknowledgements and who did what</div>

This analysis stands entirely on other people's public work. Thank you in particular to **goodpjw2008**, whose
notebook series published a nested OOF of a top-band stack, a 70-member OOF bank on one shared split, and ladder
notebooks that list **every probe with its public score**, including the ones that lost; to **Areeb Ali**, whose
*Three Ladders* notebooks publish the full version history and a CV check of each step; and to **amanatar**,
**Moran Z**, **megayak**, **masayakawamata**, **daoviet** and **Chris Deotte** for the files and ideas the ladders are
built from. Without this level of openness nobody could check the public board against CV. The full list with links
and licences is in [section 13](#s13). If we misattributed anything, please tell us in the comments.

<table style="font-size: 13px; border-collapse: collapse;">
<tr style="background: #eef4fc;"><th style="text-align: left; padding: 5px 8px;">Who</th><th style="text-align: left; padding: 5px 8px;">What this notebook uses</th></tr>
<tr><td style="padding: 5px 8px;"><b>goodpjw2008</b></td><td style="text-align: left; padding: 5px 8px;">the nested OOF of the <a href="https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298">Part 12 wide LM stack</a> (our stand-in for a test set), the <a href="https://www.kaggle.com/datasets/goodpjw2008/s6e11-oof-bank">OOF bank</a>, the <a href="https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-two-gpt-2-seeds-lb-0-95354">LM Ladder</a> probe tables, the seed-7 LM and GPT-2 seed-2026 predictions, and the <code>find()</code> / logit-rank conventions</td></tr>
<tr><td style="padding: 5px 8px;"><b>Areeb Ali</b> (areebbb)</td><td style="text-align: left; padding: 5px 8px;"><a href="https://www.kaggle.com/code/areebbb/s6e11-three-ladders-two-seeds-lb-0-95354">Three Ladders</a>: version history v9&ndash;v20 with public scores, the probe table and the CV-evidence table</td></tr>
<tr><td style="padding: 5px 8px;"><b>amanatar</b>, <b>Moran Z</b></td><td style="text-align: left; padding: 5px 8px;">the <a href="https://www.kaggle.com/datasets/amanatar/s6e11-strike10-podium-apex-095340">Strike 10</a> and <a href="https://www.kaggle.com/datasets/moranzavdi/s6e11-strike9-lb-095330">Strike 9</a> files every ladder is built on</td></tr>
<tr><td style="padding: 5px 8px;"><b>megayak</b>, <b>masayakawamata</b>, <b>daoviet</b></td><td style="text-align: left; padding: 5px 8px;"><a href="https://www.kaggle.com/datasets/megayak/s6e11-target-first-lm">target-first LM</a>, <a href="https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof">target-first GPT-2</a> and <a href="https://www.kaggle.com/datasets/daoviet/s6e11-gp-dvae-orthogonal-apex">GP + DVAE</a> OOF and test predictions</td></tr>
<tr><td style="padding: 5px 8px;"><b>Aman Atar</b> (amanatar)</td><td style="text-align: left; padding: 5px 8px;">the <a href="https://www.kaggle.com/datasets/amanatar/s6e11-rank1-q17-meta-bundle">rank-1 q17 bundle</a>: the public 0.95370 file and 16 public models with OOF and test predictions (v4)</td></tr>
<tr><td style="padding: 5px 8px;"><b>goodpjw2008</b>, <b>Areeb Ali</b> (v4)</td><td style="text-align: left; padding: 5px 8px;"><a href="https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-public-and-cv-up-lb-0-95386">Public and CV Up</a> (the replay, the three mixes and their weights) and <a href="https://www.kaggle.com/code/areebbb/s6e11-bold-then-bolder-lb-0-95388">Bold, Then Bolder</a> (the 0.95388 file, its weights and its eight probes)</td></tr>
<tr><td style="padding: 5px 8px;"><b>mathsonchain</b>, <b>Sameer Shaikh</b> (v4)</td><td style="text-align: left; padding: 5px 8px;"><a href="https://www.kaggle.com/datasets/mathsonchain/s6e11-fraud-oof-test-17-single-models">17 single models with OOF</a>; the published probes on top of the 0.95388 file</td></tr>
<tr><td style="padding: 5px 8px;"><b>this notebook</b></td><td style="text-align: left; padding: 5px 8px;">the noise measurements, the one-line noise rule, the winner's-curse simulations, the CV-tuning demos, the v4 replay checks, give-back estimates, error analysis, the nested extension gate and all figures (asteriospanoras)</td></tr>
</table>

<a id="setup"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">Setup &middot; public inputs, checked byte for byte</div>

Every input is a public file: the competition data, public OOF/test predictions and two public ladder base files.
Each one is found anywhere under `/kaggle/input` by name and checked by its SHA-256 (the `find()` idea is
goodpjw2008's). The notebook runs on CPU in about ten minutes and never trains a big model.

In [ ]:
# Imports, input discovery by SHA-256, the shared scale for predictions, and a fast AUC for repeated random splits.
import os, time, hashlib, warnings, math
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import rankdata, norm
from scipy.special import logit
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
T0 = time.time()
def log(*a): print(f"[{(time.time() - T0) / 60:4.1f} min]", *a, flush=True)

INPUT = Path(os.environ.get("KAGGLE_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("KAGGLE_WORKING", "/kaggle/working"))
_sha = {}
def sha256(p):
    p = Path(p)
    if p not in _sha: _sha[p] = hashlib.sha256(p.read_bytes()).hexdigest()
    return _sha[p]

def find(names, digest=None, inside=None, required=True):
    """A file anywhere under /kaggle/input, by (one of its) names and, if given, its SHA-256.
    If no file has the expected hash but one has the name, use it and say so (the author may have updated it)."""
    names = [names] if isinstance(names, str) else names
    hits = [p for n in names for p in sorted(INPUT.rglob(n)) if inside is None or inside in str(p)]
    good = [p for p in hits if digest is None or sha256(p) == digest]
    if good: return good[0]
    if hits:
        print(f"  NOTE: {names[0]} found but its SHA-256 differs from the one this notebook was written with; numbers may differ slightly.")
        return hits[0]
    if required: raise FileNotFoundError(names)
    print(f"  NOTE: {names[0]} not attached; the cells that need it are skipped.")
    return None

def L(v):
    """Logit rank: AUC only sees order, so every prediction goes on one scale (as the public ladders do)."""
    return logit((rankdata(v) - 0.5) / len(v))

# daoviet removed his GP + DVAE OOF/test files from his dataset on 13 Nov; Aman Atar's q17 bundle carries the same
# predictions (identical ranks, checked below by a hash of the ranks), so they are read from there when needed.
GPD_IN_BUNDLE = {"oof_ours.npy": ("oof_daoviet_gpdvae", "1c7c7fa5e49aaf8f6f63f40bb258bdff864dd49f15239fa10bb2a9c55b800688"),
                 "test_ours.npy": ("test_daoviet_gpdvae", "10f7dec317a923cc4bb67b3aff770db528abd330b53da8c53be23e8b636d050d")}
def load(names, digest, required=True):
    nm = names if isinstance(names, str) else names[0]
    if nm in GPD_IN_BUNDLE and not any(sha256(q) == digest for q in INPUT.rglob(nm)):
        key, rank_sha = GPD_IN_BUNDLE[nm]
        v = np.load(find("rank1_q17_streams.npz", "0fe4ab1263a673c8065d7145590b92da440263161bf860ce45f2976b5c4fa56f"), allow_pickle=False)[key].astype(np.float64)
        assert hashlib.sha256(rankdata(v).tobytes()).hexdigest() == rank_sha, "bundle copy of GP + DVAE differs in rank order"
        print(f"  {nm}: read from Aman Atar's q17 bundle ({key}); rank order identical to daoviet's original file")
        return v
    p = find(names, digest, required=required)
    return None if p is None else np.load(p, allow_pickle=False).astype(np.float64)

def auc_prep(score):
    """Pre-sort a score once: the negatives in score order, and for every positive the number of negatives below it."""
    o = np.argsort(score, kind="stable"); yo = y[o]
    neg_o = o[~yo].astype(np.int32); pos_rows = o[yo].astype(np.int32)
    kpos = np.cumsum(~yo)[yo].astype(np.int32)
    return neg_o, pos_rows, kpos

def auc_counts(prep, counts, denom):
    """AUC on a bootstrap sample (counts[i] = how often row i was drawn): O(n) per sample, no sort (ties ignored)."""
    neg_o, pos_rows, kpos = prep
    cn = np.concatenate(([0], np.cumsum(counts[neg_o], dtype=np.int64)))
    return float((counts[pos_rows].astype(np.int64) * cn[kpos]).sum()) / denom

def auc_sort(yr, s):
    """Plain AUC by one sort (Mann-Whitney), ~3x faster than sklearn on 700k rows; ties ignored."""
    idx = np.flatnonzero(yr[np.argsort(s, kind="stable")]); npos = len(idx)
    return (idx.sum() - npos * (npos - 1) / 2) / (npos * (len(yr) - npos))

COMP = "playground-series-s6e11"
train = pd.read_csv(find("train.csv", inside=COMP))
test_ids = pd.read_csv(find("test.csv", inside=COMP), usecols=["id"])["id"].to_numpy()
sample = pd.read_csv(find("sample_submission.csv", inside=COMP))
y = train["is_fraud"].to_numpy().astype(bool)
FOLDS = [b for _, b in StratifiedKFold(5, shuffle=True, random_state=42).split(y, y)]   # the split every public OOF uses
N, NT = len(y), len(test_ids)
log(f"train {N:,} rows, fraud rate {y.mean():.4%}; test {NT:,} rows")

In [ ]:
# One style for every figure: a colorblind-checked palette with fixed roles, recessive axes, titles that state the
# takeaway, KPI tiles, and the helpers that place text without clutter.
from matplotlib.patches import FancyBboxPatch, Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, MaxNLocator
INK, INK2, MUTED, RULE, SURF, PAGE, NAVY = "#0b0b0b", "#52514e", "#8a8984", "#d6d5d0", "#fcfcfb", "#f9f9f7", "#0b1f4b"
C = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
PUB, PRIV, CV, BAND = C[0], C[1], C[2], "#e6e5df"      # fixed roles: public = blue, private = orange, CV = aqua
CVTXT = "#0f7a55"                                        # darker aqua for text next to aqua marks
plt.rcParams.update({"figure.facecolor": PAGE, "axes.facecolor": SURF, "savefig.facecolor": PAGE, "axes.edgecolor": "#c3c2b7",
                     "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True,
                     "grid.color": "#e8e7e3", "grid.linewidth": 0.7, "axes.spines.top": False, "axes.spines.right": False,
                     "font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 11.5, "axes.titleweight": "bold",
                     "axes.titlecolor": INK, "axes.titlelocation": "left", "axes.titlepad": 10, "lines.linewidth": 2,
                     "legend.frameon": False, "axes.axisbelow": True, "figure.dpi": 110, "xtick.labelsize": 9, "ytick.labelsize": 9})
E5 = FuncFormatter(lambda v, _: f"{v:+g}" if v else "0")    # axis ticks in units of 1e-5 AUC

def headline(fig, title, sub=None, top_pad=0.0):
    """Figure title that states the takeaway (left aligned), with an optional grey subtitle."""
    h = fig.get_figheight()
    fig.text(0.012, 1 - 0.12 / h, title, ha="left", va="top", fontsize=14, fontweight="bold", color=INK)
    if sub: fig.text(0.012, 1 - 0.46 / h, sub, ha="left", va="top", fontsize=10, color=INK2)
    fig.tight_layout(rect=(0, 0, 1, 1 - ((0.80 if sub else 0.50) + top_pad) / h))

def tile(fig, rect, label, value, note, color=NAVY):
    """A KPI tile: label, big value, small note, with a coloured accent bar."""
    ax = fig.add_axes(rect); ax.set_axis_off()
    ax.add_patch(FancyBboxPatch((0, 0), 1, 1, boxstyle="round,pad=0,rounding_size=0.08", transform=ax.transAxes,
                                facecolor=SURF, edgecolor="#e6e5df", lw=1.2))
    ax.add_patch(FancyBboxPatch((0, 0.10), 0.018, 0.80, boxstyle="round,pad=0,rounding_size=0.01", transform=ax.transAxes,
                                facecolor=color, edgecolor="none"))
    ax.text(0.07, 0.76, label, transform=ax.transAxes, fontsize=9.5, color=INK2, va="center")
    ax.text(0.07, 0.45, value, transform=ax.transAxes, fontsize=20, fontweight="bold", color=INK, va="center")
    ax.text(0.07, 0.15, note, transform=ax.transAxes, fontsize=8.5, color=MUTED, va="center")

def panel_title(ax, title, sub=None):
    ax.set_title(title, loc="left", fontsize=11.5, fontweight="bold", color=INK, pad=22 if sub else 8)
    if sub: ax.text(0, 1.02, sub, transform=ax.transAxes, fontsize=9, color=INK2, va="bottom")

def zero_line(ax, axis="y"):
    (ax.axhline if axis == "y" else ax.axvline)(0, color="#b9b8b3", lw=1, zorder=1)

**How the numbers below are made.** The test labels are hidden, so we use the next best thing: the **nested OOF of
goodpjw2008's public wide LM stack** (Part 12, AUC 0.9526 on 699,960 training rows). It is a real top-band
prediction with honest out-of-fold scores. We draw hundreds of random public parts (~60k rows) and private parts
(~240k rows) from it, as two parts of a fresh test set would be drawn, and watch how scores move. Every ladder step that has public OOF files
behind it is applied to the same OOF, exactly as the ladder authors did in their own CV checks.

In [ ]:
# Load the public OOF predictions (train side). W = the nested OOF of the public wide LM stack.
OOF_FILES = {   # key: (file name(s), SHA-256, who published it)
    "W":   ("wide_lm_stack_nested.npy", "69375871698f1adf2b4c666fc7f5180edf00ac6836668353b8e7e4ca43eb2722", "goodpjw2008, wide LM stack (nested OOF)"),
    "LM":  (["lm_oof.npy", "megalm_oof_sha181c.npy"], "181ceac8f3a48fcf9a1702e1222ee13178f0688a28c1aff22c8f881532ea981a", "megayak, target-first LM"),
    "L7F": ("lm7_oof_5fold.npy", "c56d6b7282e7ea8c4c5927ac27e8b286d539ffbd66d5795564323c8b92b65242", "goodpjw2008, compact LM seed 7"),
    "GPD": ("oof_ours.npy", "91be476ddb00fe1232ba98b9a30f3c6215e81769499e3594058e4e99703c706a", "daoviet, GP + DVAE"),
    "MLP": ("oof_gbn_bag5_w512.npy", "e8ef910425641b7be8f3bf8f964b1637e9fa39937a772df56ef0cb660d3776f3", "goodpjw2008, OOF bank MLP bag"),
    "GB":  ("oof_gb_bag5.npy", "4b4bc47571ee389a54a7e70d6ac2d716207c329a1ca619a37e577de1f30aea40", "goodpjw2008, OOF bank GB bag"),
    "MS":  ("oof.npy", "3468aaaeb7851739ec0de1e06b780da6d2777fcaeef638a6852c56e68e67a206", "masayakawamata, target-first GPT-2"),
    "G26": ("gpt2_s2026_oof_5fold.npy", "4a902ecd3d725d21123f5b6239ac2190a9fd94f3de37e3157be391fcf3fe996a", "goodpjw2008, GPT-2 seed 2026"),
}
O = {k: L(load(n, h)) for k, (n, h, _) in OOF_FILES.items()}
W = O["W"]
def cv(s): return np.mean([roc_auc_score(y[b], s[b]) for b in FOLDS])
def folds_up(s, base): return sum(roc_auc_score(y[b], s[b]) > roc_auc_score(y[b], base[b]) for b in FOLDS)
CV_W = cv(W)
for k in O: print(f"  {k:4s} OOF AUC {roc_auc_score(y, O[k]):.6f}   {OOF_FILES[k][2]}")
log(f"wide LM stack, nested CV {CV_W:.6f}")

# The OOF bank's 20 'core' members (one per recipe), used as extra probe directions and for the hill-climbing demo.
bank_dir = find("manifest.csv", inside="oof-bank").parent
bank_man = pd.read_csv(bank_dir / "manifest.csv")
CORE = [n for n in bank_man.loc[bank_man["core"], "name"] if (bank_dir / f"oof_{n}.npy").exists()]
BANK = {n: L(np.load(bank_dir / f"oof_{n}.npy", allow_pickle=False).astype(np.float64)) for n in CORE}
log(f"OOF bank: {len(BANK)} core members loaded")

In [ ]:
# Every public probe that has OOF files behind it, applied to W as a move  W + w * direction.
# 'pub' = the public-LB change the authors report for that single step (x 1e-5), with the source.
MOVES = [  # (short name, direction on the OOF, reported public change, source)
    ("+0.12 megayak LM",          0.12 * (O["LM"] - W),                       +1, "goodpjw2008 v1: 0.95342 -> 0.95343"),
    ("+0.06 seed-7 LM",           0.06 * (O["L7F"] - W),                      +2, "goodpjw2008 v1: 0.95343 -> 0.95345"),
    ("+0.12 public GPT-2",        0.12 * (O["MS"] - W),                      -14, "goodpjw2008 v1 probe: -14 points"),
    ("+0.05 GP + DVAE",           0.05 * (O["GPD"] - W),                      +3, "daoviet: 0.95348 -> 0.95351"),
    ("LM weight 0.12 -> 0.09",   -0.03 * (O["LM"] - W),                       +1, "Areeb Ali v19: 0.95347 -> 0.95348"),
    ("more of both LMs",          0.04 * (O["LM"] - W) + 0.03 * (O["L7F"] - W), -2, "Areeb Ali probe: 0.95347 -> 0.95345"),
    ("less seed-7 LM",           -0.02 * (O["L7F"] - W),                       0, "Areeb Ali probe: 0.95347 -> 0.95347"),
    ("GPT-2 seed swap 0.06",      0.06 * (O["G26"] - O["MS"]),                +2, "goodpjw2008 v2: 0.95352 -> 0.95354"),
    ("GPT-2 seed swap 0.10",      0.10 * (O["G26"] - O["MS"]),                +2, "goodpjw2008 v2 probe: 0.95354"),
]
TREE = -0.10 * (O["MLP"] - W) - 0.05 * (O["GB"] - W)        # Areeb Ali's full tree move (v15); no clean single-step public number
# A family of plausible probes: a small step towards (or away from) each bank member and each public LM.
rng = np.random.default_rng(11)
FAMILY = []
for name, X in list(BANK.items()) + [("megayak LM", O["LM"]), ("seed-7 LM", O["L7F"]), ("public GPT-2", O["MS"]),
                                     ("GPT-2 seed 2026", O["G26"]), ("GP + DVAE", O["GPD"])]:
    w = rng.choice([-0.10, -0.06, -0.03, 0.03, 0.06, 0.10])
    FAMILY.append((f"{w:+.2f} {name}", w * (X - W)))
SCORES = [("base", np.zeros(N))] + [(m[0], m[1]) for m in MOVES] + [("tree move", TREE)] + FAMILY
log(f"{len(SCORES) - 1} moves to score")

RHO, TRUE, UP = {}, {}, {}
RW = rankdata(W)
for name, d in SCORES[1:]:
    s = W + d
    RHO[name] = np.corrcoef(rankdata(s), RW)[0, 1]
    TRUE[name] = (cv(s) - CV_W) * 1e5
    UP[name] = folds_up(s, W)
log("nested-CV change of every move computed")

In [ ]:
# Random public/private splits: the public part is a bootstrap sample of NPUB rows from the OOF, the private part an
# independent bootstrap sample of the other NT - NPUB rows (independent draws, as two parts of a fresh test set would be).
R = int(os.environ.get("NB_DRAWS", 300))
NPUB = int(round(0.2 * NT))
preps = [auc_prep(W + d) for _, d in SCORES]
AUC = np.zeros((len(SCORES), R, 2))
rng = np.random.default_rng(2026)
for r in range(R):
    cp = np.bincount(rng.integers(0, N, NPUB), minlength=N).astype(np.int32)
    cq = np.bincount(rng.integers(0, N, NT - NPUB), minlength=N).astype(np.int32)
    dp = float(cp[y].sum()) * float(cp[~y].sum()); dq = float(cq[y].sum()) * float(cq[~y].sum())
    for i, pr in enumerate(preps):
        AUC[i, r, 0] = auc_counts(pr, cp, dp); AUC[i, r, 1] = auc_counts(pr, cq, dq)
    if r % 100 == 99: log(f"  {r + 1} of {R} splits")
del preps
DELTA = (AUC[1:] - AUC[:1]) * 1e5           # (move, draw, public/private) change vs the base file, in 1e-5
NAMES = [n for n, _ in SCORES[1:]]
SD_PUB = DELTA[:, :, 0].std(1); SD_PRV = DELTA[:, :, 1].std(1)
AGREE = (np.sign(DELTA[:, :, 0]) == np.sign(DELTA[:, :, 1])).mean(1)
ONE_RHO = np.array([1 - RHO[n] for n in NAMES])
K_RULE = np.exp(np.mean(np.log(SD_PUB * 1e-5 / np.sqrt(ONE_RHO))))   # sigma_public(delta) ~= K_RULE * sqrt(1 - rho)
SD_BASE_PUB, SD_BASE_GAP = AUC[0, :, 0].std(), (AUC[0, :, 0] - AUC[0, :, 1]).std()
log(f"one file: public AUC SD {SD_BASE_PUB:.5f}, public-minus-private SD {SD_BASE_GAP:.5f}; rule K = {K_RULE:.4f}")

In [ ]:
# Rebuild the public ladder files on the TEST set from the authors' public inputs (formulas copied from their notebooks),
# and measure how different consecutive rungs are (rank correlation). The last file is checked byte for byte against
# the published 0.95354 output (Areeb Ali's notebook asserts the same SHA-256).
TEST_FILES = {
    "A":   ("submission_095340.csv", "12b525b84f29e32d6511e9d6105f6dc5eac79ae41b10140eed3353ea57afc159"),   # Strike 10, amanatar
    "F":   ("submission_095330.csv", "f4a7a4f8636c99fabd82945c7d81956c5559c609dcb55cde6d4e03f80b0ea105"),   # Strike 9, Moran Z
    "LM":  (["lm_test.npy", "megalm_test_sha3168.npy"], "3168779be644a530c6bdc4c141cae168bfc2feb01ac21090c917cc978a7687d3"),
    "L7F": ("lm7_test_5fold.npy", "68639fec79f081e97d40185443fa3903c9db3e62698abb01dfe31dcee217462a"),
    "GPD": ("test_ours.npy", "96f3bdde816e604438e61eac9722f7a7402f1c39bb8efe8999793b23f91d02f2"),
    "MLP": ("test_gbn_bag5_w512.npy", "035d6544d7c03108fc42ee296a8660a45825e3b1b19b7cabce1c3f42cac4e7cb"),
    "GB":  ("test_gb_bag5.npy", "7d54e5f17e0707d88fec1a2cccb05885978ae6cc2a595c5f4575c7b7d45a9bb8"),
    "MS":  ("test.npy", "6c27aff7be744f7a0cc17809e0cd04fcdea0664234af88e3040a25235fb67906"),
    "G26": ("gpt2_s2026_test_f0.npy", "40e08058ed7632ee328c2fe7d97040b856b36d2636f32cc6d2781ec9db080695"),
}
T = {}
for k, (n, h) in TEST_FILES.items():
    if isinstance(n, str) and n in GPD_IN_BUNDLE: T[k] = L(load(n, h)); continue
    p = find(n, h, required=False)
    if p is None: continue
    if str(p).endswith(".csv"):
        d = pd.read_csv(p); assert (d["id"].to_numpy() == sample["id"].to_numpy()).all(); T[k] = L(d["is_fraud"].to_numpy())
    else: T[k] = L(np.load(p, allow_pickle=False).astype(np.float64))
HAVE_LADDER = len(T) == len(TEST_FILES)
RUNGS = []   # (label, test-file score, reported public LB)
if HAVE_LADDER:
    A, F = T["A"], T["F"]
    a2 = A + 0.6 * (A - F)
    a3 = a2 + 0.12 * (T["LM"] - F)
    s352 = A + 0.6 * (A - F) + 0.09 * (T["LM"] - F) + 0.09 * (T["L7F"] - F) + 0.05 * (T["GPD"] - F) - 0.05 * (T["MLP"] - F) - 0.025 * (T["GB"] - F)
    s354 = L(s352) + 0.06 * (T["G26"] - T["MS"])
    s354b = L(s352) + 0.10 * (T["G26"] - T["MS"])
    out = (rankdata(s354) - 0.5) / NT
    chk = WORK / "rebuild_095354.csv"; sample[["id"]].assign(is_fraud=out).to_csv(chk, index=False)
    print("rebuilt 0.95354 file identical to the published one:", sha256(chk) == "1e2e72fa26309ce630a19e0df780ae63d525901f33dc23f9058581ced5e9d6e0")
    RUNGS = [("Strike 9 (Moran Z)", F, 95330), ("Strike 10 (amanatar)", A, 95340), ("+ 0.6 (S10 - S9)", a2, 95342),
             ("+ 0.12 megayak LM", a3, 95343), ("0.95352 file (several steps)", s352, 95352),
             ("+ 0.06 GPT-2 seed swap", s354, 95354)]
    rk = {lab: rankdata(s) for lab, s, _ in RUNGS}
    STEPS = []   # (label, public gain e-5, 1 - rho vs previous rung, predicted public noise SD e-5)
    for (l0, _, p0), (l1, _, p1) in zip(RUNGS[:-1], RUNGS[1:]):
        one_rho = 1 - np.corrcoef(rk[l0], rk[l1])[0, 1]
        STEPS.append((l1, p1 - p0, one_rho, K_RULE * np.sqrt(one_rho) * 1e5))
    one_rho = 1 - np.corrcoef(rankdata(s352), rankdata(s354b))[0, 1]
    STEPS_EXTRA = ("0.10 seed swap vs 0.95352 file", 2, one_rho, K_RULE * np.sqrt(one_rho) * 1e5)
    for s in STEPS: print(f"  {s[0]:32s} public {s[1]:+3d}e-5   1-rho {s[2]:.2e}   noise SD of the public gap ~{s[3]:.1f}e-5")
    del rk
else:
    print("ladder inputs not all attached; the rung figure uses the OOF moves only")
SIG_RUNG = float(np.median([s[3] for s in STEPS])) if HAVE_LADDER else float(np.median(SD_PUB))
log(f"typical noise of one rung on the public board: {SIG_RUNG:.1f}e-5")

In [ ]:
# Winner's curse, two ways.
# (1) Textbook: K probes with zero true gain, public noise SD sigma: the best public score is sigma * E[max of K normals].
KS = np.arange(1, 101)
_z = np.linspace(-6, 8, 4001)
_trap = getattr(np, "trapezoid", None) or np.trapz
EMAX = np.array([_trap(_z * k * norm.pdf(_z) * norm.cdf(_z) ** (k - 1), _z) for k in KS])
# (2) On real data: choose the best of k probes from the 'family' (and the ladder moves) by the public part of each
#     random split, then read the private part of the same split for the chosen probe.
rng = np.random.default_rng(7)
KGRID = [1, 2, 3, 5, 8, 12, 20, 30]
ALLI = np.arange(len(NAMES))
CURSE = {k: [] for k in KGRID}
for r in range(R):
    for k in KGRID:
        for _ in range(4):
            idx = rng.choice(ALLI, k, replace=False)
            j = idx[np.argmax(DELTA[idx, r, 0])]
            CURSE[k].append((DELTA[j, r, 0], DELTA[j, r, 1], TRUE[NAMES[j]]))
CURSE = {k: np.array(v) for k, v in CURSE.items()}
for k in KGRID:
    c = CURSE[k]; print(f"  best of {k:2d}: public {c[:, 0].mean():+5.1f}e-5   private {c[:, 1].mean():+5.1f}e-5   true (CV) {c[:, 2].mean():+5.1f}e-5")
# A frozen snapshot of the public leaderboard (2026-11-12 12:11 UTC): scores >= 0.95200 in units of 1e-5 above 0.95,
# and each team's number of submissions. Team names are left out on purpose.
LB_TOP = np.array([int(v) for v in """370,369,369,369,364,364,363,363,360,357,356,355,355,355,355,354,354,354,354,354,354,354,354,354,354,354,354,354,354,354,354,354,353,352,352,352,352,351,351,351,351,351,351,351,351,350,350,349,349,349,349,349,349,348,348,348,348,348,348,348,348,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,346,346,346,346,346,346,346,346,346,346,346,346,346,346,346,346,346,345,345,345,345,344,344,344,344,343,343,343,343,343,343,343,342,340,340,340,340,339,339,339,337,337,337,335,334,334,333,333,333,332,331,331,331,331,331,331,331,331,331,331,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,328,327,326,323,322,322,322,320,319,317,316,316,315,314,313,313,311,310,310,310,309,309,308,307,307,307,306,306,305,305,305,305,303,303,302,302,301,301,300,300,300,300,300,300,300,300,300,300,300,300,300,300,299,298,298,298,298,298,297,295,295,295,294,294,294,293,291,291,291,290,288,288,288,287,286,286,285,285,285,285,285,285,285,285,285,285,285,285,285,285,285,284,280,278,277,277,276,275,275,274,273,273,271,268,267,267,267,266,266,265,265,263,262,262,260,259,258,258,258,258,256,256,255,255,255,254,251,250,249,249,247,247,246,246,245,245,244,243,243,242,242,241,241,240,239,239,239,239,239,239,239,239,239,239,239,239,239,239,238,238,238,238,237,237,236,235,234,234,234,233,233,231,231,231,229,229,229,227,226,225,224,224,224,222,222,222,222,222,218,217,217,216,213,213,213,212,212,212,211,210,209,209,209,208,206,206,205,205,203,203,202,201,201,200,200,200""".split(",")])
LB_SUBS = np.array([int(v) for v in """101,31,111,102,10,55,93,18,111,2,85,20,18,20,1,96,60,10,42,96,20,11,20,10,104,17,10,13,5,2,118,13,58,20,11,84,48,91,17,66,47,17,27,18,7,48,6,10,51,39,24,15,12,4,90,9,65,10,71,4,14,95,19,30,6,57,66,5,18,8,7,1,26,5,11,34,9,2,7,14,24,64,7,11,7,18,4,10,36,10,43,15,17,6,8,7,12,6,102,45,10,27,16,1,14,8,1,20,12,38,8,7,5,15,13,22,27,42,8,120,11,42,15,12,7,23,73,73,10,12,11,30,29,10,23,19,98,78,22,31,6,2,13,11,8,16,5,8,10,2,11,7,2,11,7,7,4,15,80,27,21,2,13,8,2,6,9,8,9,5,11,10,15,3,3,7,30,40,5,33,5,4,14,26,13,5,13,46,18,23,7,10,28,28,12,16,51,1,9,30,79,6,2,23,17,11,6,30,38,31,21,51,36,22,47,6,25,16,5,3,4,4,2,1,19,10,2,3,28,19,16,1,32,2,8,77,30,16,6,5,6,5,21,31,40,3,39,4,34,10,5,26,10,2,9,11,3,2,10,7,12,5,2,9,1,3,14,3,69,14,19,15,58,3,4,120,16,100,11,10,4,3,5,7,1,9,27,79,5,12,4,78,14,3,9,4,12,5,1,8,6,16,3,5,9,20,38,1,6,5,33,5,9,5,5,25,12,10,17,1,59,6,4,6,12,2,16,2,9,26,4,36,55,31,16,3,9,9,25,15,6,19,1,5,10,4,8,11,24,5,4,30,5,16,32,7,33,39,5,1,1,1,16,9,13,4,7,7,9,7,1,9,19,3,4,1,3,11,3,4,3,7,25,5,13,7,30,31,6,41,7,7""".split(",")])
LB_TEAMS = 1577
TOP_SCORE = LB_TOP.max()
N_2SIG = int((LB_TOP >= TOP_SCORE - 2 * SIG_RUNG).sum())
log(f"leaderboard snapshot: {len(LB_TOP)} teams >= 0.95200; {N_2SIG} teams within 2 rung-noise SDs of #1")

**New in v4: the inputs of the two new ladders.** Both build on Aman Atar's public *rank-1 q17 bundle*: the public
0.95370 file plus 16 public models with out-of-fold (OOF) and test predictions on the shared five folds. goodpjw2008's
*Public and CV Up* replays the 0.95370 file on OOF rows and moves it towards 87 public members (the 16 bundle models,
the wide LM stack, 70 OOF-bank models). Areeb Ali's *Bold, Then Bolder* takes half a step further along one of
those moves. The cells below rebuild both from the public files and score every step on the same OOF rows.

In [ ]:
# v4 inputs: the q17 bundle (Aman Atar), the wide LM stack's test side, and the two LMs' test files (all checked by SHA-256).
QZ = np.load(find("rank1_q17_streams.npz", "0fe4ab1263a673c8065d7145590b92da440263161bf860ce45f2976b5c4fa56f"), allow_pickle=False)
FOLD_ID = np.zeros(N, np.int8)
for k_, b_ in enumerate(FOLDS): FOLD_ID[b_] = k_
assert (QZ["y_train"].astype(bool) == y).all() and (QZ["folds"] == FOLD_ID).all(), "bundle rows or folds differ from the shared split"
QN = [str(s) for s in QZ["stream_names"]]
W_TE = L(load("wide_lm_stack_test.npy", "b37471e6c34a50715284e807d68e15e5f3ad033e72400904f27876b24041f8cc"))
LM7_TE = L(load("lm7_test_5fold.npy", "68639fec79f081e97d40185443fa3903c9db3e62698abb01dfe31dcee217462a"))
LM_TE = L(load(["megalm_test_sha3168.npy", "lm_test.npy"], "3168779be644a530c6bdc4c141cae168bfc2feb01ac21090c917cc978a7687d3"))
_bank_files = sorted(p for p in bank_dir.glob("*.npy") if p.name.startswith(("oof_", "test_")))
BANK_OK = hashlib.sha256("".join(sha256(p) for p in _bank_files).encode()).hexdigest() == "ddd0b8e8112d49846e415a4ddbead968bce6d50fb8f351785f21a8053f420538"
print("OOF bank: all 140 files identical to the ones the ladders used:", BANK_OK)

# The 0.95370 file has no OOF of its own. goodpjw2008's replay: fit it on the test rows from the 16 bundle models and the
# wide LM stack (least squares), apply the same weights to their OOF predictions, then the same 0.06 LM move.
AT = L(QZ["test_anchor_095370"])
_one = lambda X: np.column_stack([X, np.ones(len(X))])
_XT = np.column_stack([L(QZ["test_" + n]) for n in QN] + [W_TE])
_XO = np.column_stack([L(QZ["oof_" + n]) for n in QN] + [W])
_b = np.linalg.lstsq(_one(_XT), AT, rcond=None)[0]
REPLAY_R2 = 1 - ((AT - _one(_XT) @ _b) ** 2).mean() / AT.var()
A_OOF = _one(_XO) @ _b
del _XT, _XO
REF_OOF = A_OOF + 0.06 * (L(QZ["oof_lm7_5fold"]) - L(QZ["oof_megayak_5f_lm"]))
U0 = (rankdata(AT + 0.06 * (LM7_TE - LM_TE)) - 0.5) / NT
REF_TE = logit(U0)
log(f"0.95370 file replayed on OOF: R2 {REPLAY_R2:.4f}, OOF AUC {roc_auc_score(y, REF_OOF):.6f} (wide LM stack {roc_auc_score(y, W):.6f})")

In [ ]:
# The weights of the three mixes (goodpjw2008, Public and CV Up v4) and of the 0.95388 file (Areeb Ali, Bold Then Bolder v21),
# copied from their notebooks. s: a bundle model, b: an OOF-bank model, wide: the wide LM stack.
import json
RECIPES = json.loads('''{"bold": {"s:megayak_5f_lm": -0.12205919443860914, "s:masaya_gpt2": -0.06997547490341371, "s:lm7_5fold": -0.040956430057266076, "s:gpt2_seed2026": -0.012508511564926699, "s:tabpfn_slimypunk_tp1": -0.039316699160350764, "s:sadam_tabpfn_100k_tp3": -0.061677762652753675, "s:yekenot_realmlp": -0.014637185195237004, "s:alex_glm_gen": 0.004537993603062768, "s:daoviet_gpdvae": 0.1359104270990657, "s:gbn_bag5_w512": 0.015432253915041055, "s:sameer_glm10f": 0.06272836350651828, "s:sadam_v6_gb_xgb": -0.10228295915095288, "s:sadam_v6_gb_lgb": 0.06627713517089104, "s:sadam_v4_cat": 0.03126101632182668, "s:gb_bag5": 0.008855983394809901, "s:megayak_shallow_tree": 0.28129985621486264, "wide": -0.034852407295305754, "b:ebm_all_bag": -0.050611080400723436, "b:ebm_dig": -0.016392951045769617, "b:ebm_dig_bag": -0.021971929033966105, "b:ebm_dig_r7": 0.03715898221284049, "b:ebm_diglead_bag": -0.011280640758210015, "b:ebm_diglead_i25": -0.01760600397929515, "b:ebm_diglead_r11": 0.020606132176681696, "b:ebm_diglead_r13": 0.031952532024431754, "b:ebm_i25_bag": 0.019937230510505454, "b:ebm_i25_bag4": 0.04011014273129195, "b:ebm_lead4_bag": -0.016285931935351022, "b:gb_bag10": -0.017390237028207023, "b:gb_bag5": 0.026782938331671603, "b:gb_l3": -0.06609634322241796, "b:gb_ohk": -0.02882833922678926, "b:gbc_bag3": -0.012871505523924395, "b:gbe_bag3": 0.012674094986094425, "b:gbn_bag5_w128": -0.029120675300013294, "b:gbn_bag5_w512": 0.04504705483635584, "b:gbr_bag3": -0.005251606394839109, "b:gbx_bag3": -0.01901561844474828, "b:gbx_bag5": 0.006799891770028022, "b:glm_dig": 0.003088377583964293, "b:glm_dl": -0.006428517981988452, "b:glm_dl_bag": 0.01502056793927036, "b:glm_dl_s50": -0.0501984153162254, "b:glm_dl_s50_bag": -0.028630562092033966, "b:glm_ohk": -0.017070904616440657, "b:glm_ohk2": -0.004903067440865727, "b:glm_ohk2_bag": -0.029877217995289047, "b:glm_ohk_c01": 0.0003448885987200072, "b:glm_ohkl": 0.006673473442067076, "b:glm_ohkl_bag": 0.01766365615186144, "b:glm_sb": -0.01145379830324528, "b:glm_te": -0.004792258197303907, "b:lgb_dig_l4": 0.0366960138715965, "b:lgb_dig_l7": 0.01949687340453925, "b:lgb_dig_s100": -0.051481356730372616, "b:lgb_dig_s100_bag": -7.44946546564366e-05, "b:lgb_dig_s100_bag5": 0.016566984017469158, "b:lgb_dig_s100_l3": 0.039879653631481535, "b:lgb_dig_s100_l3_bag": -0.014332571073509623, "b:lgb_diglead_bag": -0.00442949281469811, "b:lgb_diglead_bag5": -0.05069793900237127, "b:lgb_diglead_l2": -0.0434403013635199, "b:lgb_diglead_l3": 0.051384825410253424, "b:lgb_diglead_l3_bag": -0.014548087315257047, "b:lgb_diglead_l4": 0.005059024340666011, "b:lgb_dk_l4": 0.010395272568768744, "b:lgb_raw_l31": 6.272373140902098e-05, "b:lgb_raw_l4": 0.006062225361634528, "b:lgb_raw_l4_cnt": -0.060074422527671846, "b:lgb_rawdig_l4": 0.012821826099948688, "b:lgb_te_l4": 0.026453964192257394, "b:lgb_te_l4_s100": 0.007166689614341422, "b:lgb_te_l4_s300": 0.009325395175060891, "b:lgb_te_l4_s5": 0.00877347694863135, "b:realmlp_allcat": -0.010498730399732318, "b:realmlp_cnt": 0.00748969325179405, "b:realmlp_dig": -0.026045115561739243, "b:realmlp_dig_s100": 0.01482032375010226, "b:realmlp_dig_s2": 0.012780340101831603, "b:realmlp_rawdig": -0.029224792823957325, "b:realmlp_te": 0.011902655678168103, "b:realmlp_te_cnt": -0.0016811066269879565, "b:realmlp_te_s2": -0.00033527900132535184, "b:realmlp_v1": 0.02099235698670545, "b:xgb_dig_d2": -0.027214651902440812, "b:xgb_raw_d3": 0.013799277379315805, "b:xgb_te_d2": -0.012792356993134408}, "balanced": {"s:megayak_5f_lm": -0.0659946888896202, "s:masaya_gpt2": -0.07115470402589986, "s:lm7_5fold": -0.029998755707584368, "s:gpt2_seed2026": 0.005917733255070143, "s:tabpfn_slimypunk_tp1": -0.016754854043217735, "s:sadam_tabpfn_100k_tp3": -0.02209424107030813, "s:yekenot_realmlp": -0.007469976912482503, "s:alex_glm_gen": 0.004294670805904037, "s:daoviet_gpdvae": 0.07254584097984837, "s:gbn_bag5_w512": -0.002874661821916413, "s:sameer_glm10f": 0.006253891711281581, "s:sadam_v6_gb_xgb": -0.035779752095592386, "s:sadam_v6_gb_lgb": 0.014771844136765163, "s:sadam_v4_cat": 0.005442134718824526, "s:gb_bag5": 0.02197944620878155, "s:megayak_shallow_tree": 0.10861837977873254, "wide": 0.008857483348604336, "b:ebm_all_bag": -0.035648523256013816, "b:ebm_dig": 0.0021571996755421373, "b:ebm_dig_bag": -0.00870932789686656, "b:ebm_dig_r7": 0.014675020864131676, "b:ebm_diglead_bag": -0.011242137972560769, "b:ebm_diglead_i25": -0.011799461220371074, "b:ebm_diglead_r11": 0.018395467162953583, "b:ebm_diglead_r13": 0.019913528869070343, "b:ebm_i25_bag": 0.014300715796457614, "b:ebm_i25_bag4": 0.014776764302437287, "b:ebm_lead4_bag": -0.010845849051441005, "b:gb_bag10": -0.01586540001387272, "b:gb_bag5": 0.013795945657627443, "b:gb_l3": -0.022653116683931083, "b:gb_ohk": -0.005825205521352985, "b:gbc_bag3": -0.006125060624374499, "b:gbe_bag3": 0.004479022738251753, "b:gbn_bag5_w128": -0.008463251429546712, "b:gbn_bag5_w512": 0.02476864383685032, "b:gbr_bag3": -9.909508786856773e-06, "b:gbx_bag3": -0.006276689543614139, "b:gbx_bag5": 0.005886331747934474, "b:glm_dig": 0.0005311280711308347, "b:glm_dl": -0.0059032327276077766, "b:glm_dl_bag": 0.014006601323714443, "b:glm_dl_s50": -0.022355265176809174, "b:glm_dl_s50_bag": 0.0006990668756181338, "b:glm_ohk": -0.005312350261809259, "b:glm_ohk2": 0.00678460176455991, "b:glm_ohk2_bag": -0.014305563020749566, "b:glm_ohk_c01": 0.002124044507600178, "b:glm_ohkl": -0.0018632849202664824, "b:glm_ohkl_bag": 0.009617257091917682, "b:glm_sb": -0.0032167242579132175, "b:glm_te": -0.0024289780895983912, "b:lgb_dig_l4": 0.010390240631489167, "b:lgb_dig_l7": 0.006166614002474661, "b:lgb_dig_s100": -0.016328847468105947, "b:lgb_dig_s100_bag": 0.006054175271612747, "b:lgb_dig_s100_bag5": 0.0032893020819732264, "b:lgb_dig_s100_l3": 0.01156644449639154, "b:lgb_dig_s100_l3_bag": -0.0072697486729145595, "b:lgb_diglead_bag": 0.003752570618535255, "b:lgb_diglead_bag5": -0.01387844815389441, "b:lgb_diglead_l2": -0.008722162257095078, "b:lgb_diglead_l3": 0.01340449460759475, "b:lgb_diglead_l3_bag": -0.009648198615136732, "b:lgb_diglead_l4": 0.0023017753746317295, "b:lgb_dk_l4": 0.00404320185129622, "b:lgb_raw_l31": 0.0004917472972874557, "b:lgb_raw_l4": -0.00011914588071201624, "b:lgb_raw_l4_cnt": -0.015582993656188236, "b:lgb_rawdig_l4": 0.001729542007007652, "b:lgb_te_l4": 0.008969005313344148, "b:lgb_te_l4_s100": 3.676618403346857e-05, "b:lgb_te_l4_s300": 0.0022842969787489017, "b:lgb_te_l4_s5": 0.002931297790132998, "b:realmlp_allcat": -0.0026909874980677114, "b:realmlp_cnt": 0.0021648947039792175, "b:realmlp_dig": -0.009016404783169642, "b:realmlp_dig_s100": 0.005402396661241892, "b:realmlp_dig_s2": 0.0022649717220251738, "b:realmlp_rawdig": -0.007755994202623359, "b:realmlp_te": 0.0032482793804713803, "b:realmlp_te_cnt": 0.000847393585301589, "b:realmlp_te_s2": 0.0004964361210193442, "b:realmlp_v1": 0.0064450941750321756, "b:xgb_dig_d2": -0.001615855509527677, "b:xgb_raw_d3": 0.00433269057718474, "b:xgb_te_d2": -0.0033627181678075714}, "careful": {"s:megayak_5f_lm": -0.08450474886996073, "s:masaya_gpt2": -0.04902561986411483, "s:lm7_5fold": -0.016891411336523863, "s:gpt2_seed2026": 0.014547738041531408, "s:tabpfn_slimypunk_tp1": -0.0123457470481866, "s:sadam_tabpfn_100k_tp3": -0.015910384586476225, "s:yekenot_realmlp": -0.003253325303861341, "s:alex_glm_gen": 0.002612752988313417, "s:daoviet_gpdvae": 0.04685747302679586, "s:gbn_bag5_w512": 0.0043515522989154235, "s:sameer_glm10f": 0.015478953160303576, "s:sadam_v6_gb_xgb": -0.028037375140335462, "s:sadam_v6_gb_lgb": 0.019533386719025193, "s:sadam_v4_cat": 0.00589878484038633, "s:gb_bag5": 0.02217844040888381, "s:megayak_shallow_tree": 0.10122717023890326, "wide": 0.08081797187087163, "b:ebm_all_bag": -0.03128848896508937, "b:ebm_dig": 0.002420259254897357, "b:ebm_dig_bag": -0.006646622819934664, "b:ebm_dig_r7": 0.010794306141448952, "b:ebm_diglead_bag": -0.0117042245417344, "b:ebm_diglead_i25": -0.008918615668229131, "b:ebm_diglead_r11": 0.017495433781515475, "b:ebm_diglead_r13": 0.019436209350096583, "b:ebm_i25_bag": 0.008522596080523918, "b:ebm_i25_bag4": 0.016476990933893144, "b:ebm_lead4_bag": -0.01130787311582097, "b:gb_bag10": -0.013583645440115318, "b:gb_bag5": 0.016684891246275074, "b:gb_l3": -0.019083828087590973, "b:gb_ohk": -0.006177878317621568, "b:gbc_bag3": -0.007269120619879105, "b:gbe_bag3": 0.004681311487245302, "b:gbn_bag5_w128": -0.005040768827170892, "b:gbn_bag5_w512": 0.023637688632069328, "b:gbr_bag3": -0.00024830818253431273, "b:gbx_bag3": -0.006400911249302103, "b:gbx_bag5": 0.0036937689629500865, "b:glm_dig": -3.7306124542630744e-05, "b:glm_dl": -0.007156160004066149, "b:glm_dl_bag": 0.008441014574504154, "b:glm_dl_s50": -0.01334934207158471, "b:glm_dl_s50_bag": 0.001844211766019183, "b:glm_ohk": -0.00562390612264448, "b:glm_ohk2": 0.008907953791620765, "b:glm_ohk2_bag": -0.015300133811741116, "b:glm_ohk_c01": -0.005892186175925242, "b:glm_ohkl": -0.00010117004854648993, "b:glm_ohkl_bag": 0.008342574170709143, "b:glm_sb": -0.003455433101783884, "b:glm_te": -0.0018554925449663707, "b:lgb_dig_l4": 0.012788668778136636, "b:lgb_dig_l7": 0.006133544143597403, "b:lgb_dig_s100": -0.01581131868317444, "b:lgb_dig_s100_bag": 0.005976683240784171, "b:lgb_dig_s100_bag5": 0.003272572328457066, "b:lgb_dig_s100_l3": 0.011358913543846424, "b:lgb_dig_s100_l3_bag": -0.007671455946372908, "b:lgb_diglead_bag": 0.0026446221720072436, "b:lgb_diglead_bag5": -0.014415491763205732, "b:lgb_diglead_l2": -0.009879458752685616, "b:lgb_diglead_l3": 0.015081200121801388, "b:lgb_diglead_l3_bag": -0.01231464394864841, "b:lgb_diglead_l4": 0.0027104104670708385, "b:lgb_dk_l4": 0.002097897271309318, "b:lgb_raw_l31": 0.0010331280960220208, "b:lgb_raw_l4": -0.0013378069288362123, "b:lgb_raw_l4_cnt": -0.013507197348019013, "b:lgb_rawdig_l4": 0.0028805514532224763, "b:lgb_te_l4": 0.008125965905514013, "b:lgb_te_l4_s100": 0.0030857860977705903, "b:lgb_te_l4_s300": 0.0022465238467168505, "b:lgb_te_l4_s5": -0.0007517378836866291, "b:realmlp_allcat": -0.00013519997284003595, "b:realmlp_cnt": 0.0031564480735426013, "b:realmlp_dig": -0.009290279722601536, "b:realmlp_dig_s100": 0.005582584293801132, "b:realmlp_dig_s2": 0.0018383277487227473, "b:realmlp_rawdig": -0.009112515110809292, "b:realmlp_te": 0.003079185734258433, "b:realmlp_te_cnt": -0.00037244665110067394, "b:realmlp_te_s2": -0.0021716481344977163, "b:realmlp_v1": 0.005001493601478614, "b:xgb_dig_d2": -0.004210955412618814, "b:xgb_raw_d3": 0.004254033686957146, "b:xgb_te_d2": -0.0023974804930285173}}''')
W388 = json.loads('''{"s:alex_glm_gen": 0.004659655001642133, "b:lgb_diglead_l3": 0.07037499081158276, "s:daoviet_gpdvae": 0.16759272015867438, "b:realmlp_v1": 0.028265988392542088, "b:gb_bag10": -0.018152655535374172, "b:lgb_dig_l7": 0.026162003105571548, "b:lgb_dig_s100_bag5": 0.023205824985217123, "s:lm7_5fold": -0.04643526723210693, "b:lgb_te_l4_s300": 0.012845944273216886, "b:lgb_diglead_bag": -0.008520524531314792, "s:masaya_gpt2": -0.06938586034217065, "b:lgb_dig_s100": -0.06905761136150596, "b:gbx_bag3": -0.025385082895315353, "b:lgb_te_l4": 0.035196443631714015, "s:sadam_v6_gb_xgb": -0.13553456267863312, "b:lgb_dig_l4": 0.04984890049165017, "b:glm_sb": -0.015572335325911311, "b:glm_dl_s50_bag": -0.043295376575860016, "b:lgb_te_l4_s100": 0.010731651329495398, "b:glm_ohk_c01": -0.0005446893557200782, "b:gbn_bag5_w512": 0.0551862603361086, "b:glm_ohk": -0.022950181793756356, "b:glm_dl_bag": 0.015527551247048318, "b:realmlp_dig_s100": 0.019529287294532443, "b:gbr_bag3": -0.007872454837865235, "wide": -0.0567073526172608, "s:gbn_bag5_w512": 0.024585711783519787, "b:xgb_dig_d2": -0.04001405009889738, "b:ebm_i25_bag4": 0.052776831945719284, "s:megayak_shallow_tree": 0.3676405944329277, "s:gb_bag5": 0.0022942519878240766, "b:lgb_dig_s100_bag": -0.0031388296177910285, "s:gpt2_seed2026": -0.02172163397492512, "b:gbe_bag3": 0.016771631110015763, "s:yekenot_realmlp": -0.018220789336614255, "s:sadam_tabpfn_100k_tp3": -0.08146952344397645, "s:sadam_v6_gb_lgb": 0.09202978068795398, "b:glm_ohk2": -0.010746902043578546, "b:gb_ohk": -0.0403299060795074, "b:ebm_dig": -0.025668026406425494, "b:ebm_diglead_r11": 0.021711464683545754, "b:xgb_te_d2": -0.017507176405797825, "b:realmlp_rawdig": -0.03995919213462431, "b:gbn_bag5_w128": -0.03944938723524659, "s:sadam_v4_cat": 0.044170457123327754, "b:gb_l3": -0.08781795649166141, "b:lgb_diglead_bag5": -0.0691076844266097, "b:ebm_diglead_r13": 0.03797203360211246, "b:glm_dl": -0.0066911606091787906, "b:ebm_all_bag": -0.05809235897307825, "b:ebm_diglead_bag": -0.011299892151034639, "b:lgb_dig_s100_l3_bag": -0.017863982273807155, "b:realmlp_te_s2": -0.0007511365624976998, "b:realmlp_cnt": 0.010152092525701466, "b:ebm_i25_bag": 0.022755487867529373, "b:lgb_dig_s100_l3": 0.054036258199026535, "b:glm_dig": 0.004367002340381022, "b:lgb_diglead_l2": -0.060799370916732313, "b:lgb_diglead_l3_bag": -0.016998031665317204, "b:lgb_rawdig_l4": 0.018367968146419205, "b:realmlp_te_cnt": -0.0029453567331327295, "b:ebm_dig_r7": 0.04840096288719489, "b:gb_bag5": 0.033276434668693684, "b:gbx_bag5": 0.007256671781074796, "b:glm_ohkl": 0.010941852623233855, "s:tabpfn_slimypunk_tp1": -0.05059762171891728, "b:glm_dl_s50": -0.06411999038593352, "s:sameer_glm10f": 0.09096559940413664, "b:ebm_diglead_i25": -0.020509275358757188, "b:lgb_te_l4_s5": 0.011694566527880524, "b:ebm_dig_bag": -0.02860322960251588, "b:lgb_raw_l31": -0.00015178805153019635, "b:glm_ohk2_bag": -0.03766304548255879, "b:lgb_raw_l4": 0.0091529109828078, "b:xgb_raw_d3": 0.018532570780381338, "b:ebm_lead4_bag": -0.01900597337730603, "s:megayak_5f_lm": -0.1500914472131036, "b:glm_te": -0.005973898251156665, "b:glm_ohkl_bag": 0.021686855681833317, "b:realmlp_allcat": -0.014402601850564621, "b:lgb_raw_l4_cnt": -0.08232013696341364, "b:realmlp_dig": -0.034559470951024046, "b:lgb_dk_l4": 0.013571307927505006, "b:realmlp_dig_s2": 0.018038024291734817, "b:lgb_diglead_l4": 0.006437648823683151, "b:realmlp_te": 0.016229843827016463, "b:gbc_bag3": -0.016244727973699344}''')
assert set(W388) == set(RECIPES["bold"]) and max(abs(W388[k] - (1.5 * RECIPES["bold"][k] - 0.5 * RECIPES["balanced"][k])) for k in W388) < 1e-12

def qmember(k, part):
    if k.startswith("s:"): return L(QZ[f"{part}_{k[2:]}"])
    if k.startswith("b:"): return L(np.load(bank_dir / f"{part}_{k[2:]}.npy", allow_pickle=False).astype(np.float64))
    return W if part == "oof" else W_TE
QM_O = {k: qmember(k, "oof") for k in RECIPES["bold"]}
QM_T = {k: qmember(k, "test") for k in RECIPES["bold"]}
def mix(wts, part):
    """(1 - sum w) base + sum_k w_k member_k, summed in the authors' order (so the test file is reproduced bit for bit)."""
    M, base = (QM_O, REF_OOF) if part == "oof" else (QM_T, REF_TE)
    return (1 - sum(wts.values())) * base + sum(w * M[k] for k, w in wts.items())
MO = {r: mix(RECIPES[r], "oof") for r in ("careful", "balanced", "bold")}
MT = {r: mix(RECIPES[r], "test") for r in ("careful", "balanced", "bold")}
MO["388"], MT["388"] = mix(W388, "oof"), mix(W388, "test")
S388 = (rankdata(MT["388"]) - 0.5) / NT
BEST_SHA = "b097969454128762dcd4929116d08fc9f00b3a6ebfe65fcbf7979ffaea3d729b"
_chk = WORK / "rebuild_095388.csv"; sample[["id"]].assign(is_fraud=S388).to_csv(_chk, index=False)
REBUILD_OK = sha256(_chk) == BEST_SHA
print("rebuilt 0.95388 file identical to the published one (SHA-256):", REBUILD_OK)
if not REBUILD_OK:
    print("  NOTE: floating-point differences between numpy builds can swap a few tied ranks; the file is the same recipe.")

# Every probe of the two ladders is a linear combination of these mixes (the weights are linear), on OOF and on test alike.
def combo(parts, side):
    D, base = (MO, REF_OOF) if side == "oof" else (MT, REF_TE)
    return sum(c * (base if k == "ref" else D[k]) for k, c in parts.items())
PROBES = {   # name: (combination, public LB, author)
    "careful": ({"careful": 1}, 95376, "goodpjw2008"), "balanced": ({"balanced": 1}, 95380, "goodpjw2008"),
    "bold": ({"bold": 1}, 95386, "goodpjw2008"),
    "1.25 x bold": ({"bold": 1.25, "ref": -0.25}, 95387, "Areeb Ali"), "1.5 x bold": ({"bold": 1.5, "ref": -0.5}, 95387, "Areeb Ali"),
    "bold + 0.5 (bold - careful)": ({"bold": 1.5, "careful": -0.5}, 95387, "Areeb Ali"),
    "bold + 0.5 (bold - balanced)": ({"bold": 1.5, "balanced": -0.5}, 95388, "Areeb Ali"),
    "bold + 1.0 (bold - balanced)": ({"bold": 2.0, "balanced": -1.0}, 95387, "Areeb Ali"),
    "bold + 1.5 (bold - balanced)": ({"bold": 2.5, "balanced": -1.5}, 95387, "Areeb Ali"),
    "bold + 2.0 (bold - balanced)": ({"bold": 3.0, "balanced": -2.0}, 95385, "Areeb Ali"),
    "1.25 x (bold + 1.0 (bold - balanced))": ({"bold": 2.5, "balanced": -1.25, "ref": -0.25}, 95386, "Areeb Ali"),
}
PO = {n: combo(c, "oof") for n, (c, _, _) in PROBES.items()}
assert np.abs(PO["bold + 0.5 (bold - balanced)"] - MO["388"]).max() < 1e-4   # same file up to float rounding
PO["bold + 0.5 (bold - balanced)"] = MO["388"]
PO["10% wide LM stack"] = 0.9 * A_OOF + 0.1 * W; PO["25% wide LM stack"] = 0.75 * A_OOF + 0.25 * W   # goodpjw2008 v3 probes: 95368, 95362
PO["388 + 5% GP + DVAE"] = 0.95 * L(MO["388"]) + 0.05 * L(QZ["oof_daoviet_gpdvae"])                   # probes on the 0.95388 file reported
PO["388 + 5% shallow tree"] = 0.95 * L(MO["388"]) + 0.05 * L(QZ["oof_megayak_shallow_tree"])         # by Sameer Shaikh (Part 3): 95385, 95387

def fold_aucs(s): return np.array([auc_sort(y[b], s[b]) for b in FOLDS])
FA = {"ref": fold_aucs(REF_OOF), "W": fold_aucs(W)}
for n, s in PO.items(): FA[n] = fold_aucs(s)
OOFAUC = {"ref": roc_auc_score(y, REF_OOF), "W": roc_auc_score(y, W)}
for n, s in PO.items(): OOFAUC[n] = roc_auc_score(y, s)
def chg(a, b):
    """CV change of a over b: overall OOF AUC change (x 1e-5, as the authors report it) and the five per-fold changes."""
    return (OOFAUC[a] - OOFAUC[b]) * 1e5, (FA[a] - FA[b]) * 1e5
for n in PROBES: print(f"  {n:40s} public {PROBES[n][1] - 95370:+3d}  in-sample CV vs 0.95370 replay {chg(n, 'ref')[0]:+5.1f}  folds up {(chg(n, 'ref')[1] > 0).sum()}/5")
# independent check: the same change from the per-fold AUCs (a different estimator; should be close)
print(f"check: bold vs replay, mean of fold changes {chg('bold', 'ref')[1].mean():+.1f}e-5; authors report +28.4 / +19.0 / +32.8 / +37.1 in-sample")

In [ ]:
# Public noise of every step (the one-line rule on the TEST files) and the private give-back of public selection,
# simulated on the OOF rows with the real public/private sizes: on each random split the probe with the best public gap
# is kept, as the authors did, and its private gap is read.
TE = {"0.95354 file": L(out) if HAVE_LADDER else None, "0.95370 file": REF_TE}
for n, (c, _, _) in PROBES.items(): TE[n] = combo(c, "test")
def rho_te(a, b): return np.corrcoef(rankdata(TE[a]), rankdata(TE[b]))[0, 1]
def sig(r): return K_RULE * np.sqrt(max(1 - r, 0)) * 1e5

FAMILIES = {   # the probes each author compared on the board before keeping one, and the base they compared against
    "goodpjw2008": ("ref", ["careful", "balanced", "bold", "10% wide LM stack", "25% wide LM stack"]),
    "Areeb Ali": ("bold", ["1.25 x bold", "1.5 x bold", "bold + 0.5 (bold - careful)", "bold + 0.5 (bold - balanced)",
                           "bold + 1.0 (bold - balanced)", "bold + 1.5 (bold - balanced)", "bold + 2.0 (bold - balanced)",
                           "1.25 x (bold + 1.0 (bold - balanced))"]),
}
_sc = {"ref": REF_OOF, **PO}
_names = ["ref"] + [n for n in PO]
_preps = [auc_prep(_sc[n]) for n in _names]
QAUC = np.zeros((len(_names), R, 2))
rng = np.random.default_rng(2611)
for r_ in range(R):
    cp = np.bincount(rng.integers(0, N, NPUB), minlength=N).astype(np.int32)
    cq = np.bincount(rng.integers(0, N, NT - NPUB), minlength=N).astype(np.int32)
    dp = float(cp[y].sum()) * float(cp[~y].sum()); dq = float(cq[y].sum()) * float(cq[~y].sum())
    for i, pr in enumerate(_preps): QAUC[i, r_, 0] = auc_counts(pr, cp, dp); QAUC[i, r_, 1] = auc_counts(pr, cq, dq)
del _preps
QI = {n: i for i, n in enumerate(_names)}
def qgap(a, b): return (QAUC[QI[a]] - QAUC[QI[b]]) * 1e5        # (draw, public/private)
GIVE = {}
for fam, (base, probes) in FAMILIES.items():
    G = np.stack([qgap(p, base) for p in probes])                 # (probe, draw, 2)
    j = G[:, :, 0].argmax(0)
    pick_pub, pick_prv = G[j, np.arange(R), 0], G[j, np.arange(R), 1]
    GIVE[fam] = dict(pub=pick_pub, prv=pick_prv, back=pick_pub - pick_prv, sd_pub=G[:, :, 0].std(1), K=len(probes))
    print(f"  {fam}: best of {len(probes)} on public: public {pick_pub.mean():+.1f}, private {pick_prv.mean():+.1f}, give-back {np.mean(pick_pub - pick_prv):+.1f}e-5")
# check of the noise rule on these new, larger steps: measured public SD vs K_RULE * sqrt(1 - rho) on the OOF
RULE_CHECK = []
for n in ["careful", "balanced", "bold"]:
    rr = np.corrcoef(rankdata(PO[n]), rankdata(REF_OOF))[0, 1]
    RULE_CHECK.append((n, 1 - rr, qgap(n, "ref")[:, 0].std(), sig(rr)))
for n in ["bold + 0.5 (bold - balanced)", "bold + 1.5 (bold - balanced)", "1.5 x bold"]:
    rr = np.corrcoef(rankdata(PO[n]), rankdata(PO["bold"]))[0, 1]
    RULE_CHECK.append((n + " vs bold", 1 - rr, qgap(n, "bold")[:, 0].std(), sig(rr)))
for n, orr, m, p in RULE_CHECK: print(f"  rule check {n:38s} 1-rho {orr:.2e}: measured public SD {m:4.1f}, rule {p:4.1f}")

In [ ]:
# Far-apart files: the public gap noise between two different strong files (the wide LM stack and the best of
# mathsonchain's single models), measured directly on the same random splits as v3.
FAR = L(load("oof_boost_j.npy", "94a822b31e8e3b132870ece2517972053e47a501ee8ec387d86ce5af164a35b5"))
_pa, _pb = auc_prep(W), auc_prep(FAR)
rng = np.random.default_rng(2612); _g = []
for r_ in range(R):
    cp = np.bincount(rng.integers(0, N, NPUB), minlength=N).astype(np.int32); dp = float(cp[y].sum()) * float(cp[~y].sum())
    _g.append((auc_counts(_pa, cp, dp) - auc_counts(_pb, cp, dp)) * 1e5)
FAR_RHO = np.corrcoef(rankdata(W), rankdata(FAR))[0, 1]; FAR_SD = float(np.std(_g))
del _pa, _pb
log(f"two different strong files (rho {FAR_RHO:.4f}): public gap SD {FAR_SD:.1f}e-5 (rule {sig(FAR_RHO):.1f})")

# Leaderboard snapshot 14 Nov 2026 01:03 UTC (scores >= 0.95200 as points above 0.95, and submission counts; no names).
LB14_TOP = np.array([int(v) for v in """784,604,442,437,432,406,400,395,394,394,393,392,391,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,388,387,387,387,386,386,386,386,386,386,385,384,384,383,382,381,376,372,372,372,371,370,370,370,370,370,370,370,370,370,370,370,370,370,370,370,370,370,370,370,369,367,364,364,364,363,361,361,361,359,357,357,357,356,356,356,355,355,355,355,355,355,354,354,354,354,354,354,354,354,354,354,354,354,354,354,354,354,354,354,354,354,353,352,351,351,351,351,351,350,350,349,349,349,348,348,348,348,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,347,346,346,346,346,346,346,346,346,346,346,346,346,345,345,345,345,344,344,343,343,343,343,343,342,341,340,340,340,339,339,337,337,334,333,333,333,331,331,331,331,331,331,331,331,331,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,330,326,325,323,322,322,322,320,319,317,316,315,313,313,311,311,310,310,310,309,309,309,308,307,307,307,306,306,305,303,303,302,302,301,300,300,300,300,300,300,300,300,300,300,300,300,300,300,299,298,298,298,298,297,296,295,295,295,295,294,294,293,291,291,291,290,290,288,288,288,287,286,286,285,285,285,285,285,285,285,285,285,285,285,285,285,285,285,285,284,280,278,277,277,276,275,275,275,274,274,274,273,268,268,268,268,267,267,267,266,266,265,265,262,262,262,262,260,258,258,258,258,257,256,256,255,255,255,254,251,250,249,249,248,247,247,246,246,245,245,244,243,243,242,242,241,241,241,240,240,239,239,239,239,239,239,239,239,239,239,239,239,239,238,238,238,238,238,237,237,237,236,235,235,234,234,233,233,231,231,231,230,230,229,229,228,226,225,224,224,224,224,222,222,222,222,222,222,219,218,217,217,216,216,213,213,212,212,211,210,209,209,209,208,206,206,205,205,203,203,203,203,202,201,201,200,200,200""".split(",")])
LB14_SUBS = np.array([int(v) for v in """131,34,61,10,42,112,117,28,81,41,31,99,120,90,12,111,107,108,5,18,113,95,18,17,2,12,11,93,6,112,7,128,28,31,1,3,8,23,2,5,7,90,9,24,28,74,67,13,22,23,16,107,110,12,8,1,47,18,10,54,102,24,68,25,2,76,30,21,18,30,65,62,1,21,9,4,1,13,1,76,20,84,11,1,20,67,19,18,20,33,18,10,55,71,14,45,46,47,57,6,29,34,17,22,6,16,22,30,12,20,7,58,11,60,12,10,5,2,14,26,21,25,2,1,3,14,9,16,6,15,7,84,18,11,19,7,22,10,6,10,39,13,90,10,4,14,30,7,8,26,5,11,2,7,15,24,9,4,15,36,15,6,8,7,9,27,16,1,14,8,1,21,12,8,7,5,15,13,43,8,130,12,42,15,12,6,31,10,13,11,30,10,23,19,31,78,6,8,8,16,5,8,10,2,11,7,2,7,7,4,15,27,21,2,13,8,2,6,13,8,9,5,11,10,10,3,3,7,7,5,10,33,5,4,14,27,13,17,13,18,7,10,7,28,28,13,16,51,1,2,9,30,87,6,2,24,11,38,31,21,51,36,47,6,25,16,5,3,4,4,2,1,19,10,2,3,28,16,1,32,2,9,5,77,30,12,16,6,6,5,21,31,40,3,28,39,4,34,10,5,26,10,2,9,11,3,2,10,7,12,5,2,3,9,1,3,14,4,82,16,19,16,58,5,3,4,116,130,13,16,89,85,29,12,4,3,5,7,1,5,9,5,5,18,14,1,14,3,9,4,12,12,5,1,8,6,19,3,5,9,20,27,38,1,6,5,33,5,9,5,5,25,12,23,10,17,5,1,59,6,4,6,12,2,16,2,9,4,36,65,31,16,2,3,9,9,25,6,15,6,19,5,5,10,8,49,12,24,5,10,17,4,30,8,32,7,33,24,39,5,1,8,1,1,16,9,3,14,4,7,7,8,9,2,19,3,4,1,3,11,3,4,3,7,25,5,13,7,6,3,30,31,6,41,7,7""".split(",")])
LB14_TEAMS = 1724
BAND14 = LB14_TOP[4]                                   # #5, the top of the dense band
print(f"14 Nov: #1 {LB14_TOP[0]}, #2 {LB14_TOP[1]}, #5 {LB14_TOP[4]}, #10 {LB14_TOP[9]}, #50 {LB14_TOP[49]}, #100 {LB14_TOP[99]} (x1e-5 above 0.95)")
print(f"#1 is {LB14_TOP[0] - BAND14} points above #5 = {(LB14_TOP[0] - BAND14) / FAR_SD:.1f} SDs of the gap noise of two DIFFERENT strong files")

In [ ]:
# THE EXTENSION (fixed before looking at any result): at most two steps on top of the 0.95388 file, each chosen by
# nested CV on public OOF only, never by the public board.
#  * candidates: a step  B + w (M - B)  in logit-rank space towards (w > 0) or away from (w < 0) one public member M
#    that has honest OOF on the shared five folds and is NOT one of the 87 members already in the mix (so the
#    mix weights, which saw all five folds, never saw M); w in {+-0.01, +-0.02, +-0.03, +-0.05};
#  * nested: for each outer fold, the four other folds choose (M, w) by their mean AUC gain; the outer fold scores it;
#  * gate: the outer-fold gain is positive on at least 4 of 5 folds AND its mean is positive. Otherwise: no step.
EXT_FILES = {   # name: (OOF file, its SHA-256, test file, its SHA-256)
    "wide stack, no LM (goodpjw2008)": ("wide_stack_nested.npy", "a829d980d77751fc5890478b036d310329293044e4a360891bac04e3e066f82b",
                                        "wide_stack_test.npy", "3284788a79e36ba45a86d364d56a2d483e1842d17f5577aa3a6010c2505c7502"),
    "EBM bag (goodpjw2008)": ("ebm_b4_oof.npy", "6597874a645e78ebcb9abec59404d15ec2a306825212e9933ab867152da68ae6",
                              "ebm_b4_test.npy", "76693a45e4e8f84b859e1622397a6362fd620cd38da7a013a4f160c6beee1ed4"),
}
for n_, (o_, t_) in {
    "boost_j": ("94a822b31e8e3b132870ece2517972053e47a501ee8ec387d86ce5af164a35b5", "61ee0764dd4245a5b750d102a448ac7ca6b3920c7e4a9d5b3f9162eb37534402"),
    "glm_j": ("46eaf6a1e007ab1b60a008de608d5338a6845e250b6447f9e41b55912960f2b9", "5f7350c1bf8c506f21754d96c4282e0b423421556ea19ee3e508cbb734b874e6"),
    "glm_spline": ("c53cad1d8fb40bd2e7812a7b061c98400d86faff922ff365215e19395208dae8", "88656f9bb1cc2c8ccf29061a06bcc0905afea54b72d40883aacd71b30bbe3204"),
    "glm_te_nc": ("e3ae0153025f2b1525d44ca42c12f27e7cc2abb41bf5b7e4c6e22072cfee8278", "4bb4a8543e6283a632fc0d6a0c6653c37e6526fe24a4cf952f66d779bb804a66"),
    "glm_te": ("3c4f35e45485e2fb2779d31873176424aa07df5094924321fef6387719a2667e", "38f0a2628498ab6f3fea2f370cc13597f1d24b567e7517f142b761b32bbe463b"),
    "glm_v2": ("878f207e3268a7ec1be73c2801429f25de82c7df85ec0106b6e81629c31d3922", "842b2fb6c89d1847856550d8667d51fa515951b6e7598155779c3214f3dd5dca"),
    "lgb_c": ("2b19902834631600ab45f7b47913e831e7e5169c0632ea7740bc7c384623b4cd", "77e80669786a648bbff36eb1b52e6f992a65a49152bcee8f9782ead8fb88d47d"),
    "lgb_fe_decimals": ("cd1a61bd42cde608412b6775e8a3865c640d24a2d36d7d2a47df0363775208ec", "f85b95d8b5f1963c0104dbd0599ef5e1312714343f0a654e53f65296505da0bb"),
    "lgb_glmboost": ("e833e12bf0d1bcd88f706eb99b69c18e7cac543862c0faaa6c7e14a6133db952", "6b1e9a7dfa451dbdbb14b76c396eea52006bc598c78a8f7ad2e3a3bf8e9e26cc"),
    "lgb_raw": ("0d35fe489e9dcf28d22095558623c2916223ff6420cbdd0c742eba48d34f498c", "ef1198cbbca9c2b46ce89ad3ccd9356372ba965f4d1cd3df28c75bf973d07d0f"),
    "lgb_teacher": ("7fe6d993ec14ddee7f38dec776de49855a86155b5b14ee359986b8aa0473b289", "6c2b5b5362e3fc1630452268c4f0e063d74c34d0e15b2c07351bccb53440bfb6"),
    "lgb_te": ("efab3416f6da7c73909d8421fd29052f3ed1ad188b2e3a4f2c40574e3b271e8b", "1763eb9f4de155814fe6ee0a3bec18728540b43cb4169cecdb090b7fca6d5ac8"),
    "lgb_v1": ("9f56e7c60647bf391625c9ffec2c9d2e06f84368843cf3f441874b71f2cac534", "94c13b2718536032a7b13218032c3d3ad7a5af055e03a8ef98d339e806a955cc"),
    "lgb_v2": ("0db99ece05c03d3134d837ee9bb1caa42db83a4a8a360b67eddf4de1aa735b91", "caa5dca66b95e87d23694e29948af18457d85d0a9e5b815f7cf4e02a2124024f"),
    "xgb_te": ("7cf18c1b63afc298b379ef3789c4dda8613df8920d700f558c4b2754ccf4e177", "abf33f3501436c23e84de8a593375ca25a0308a5ca350cf8043608ae0cfcd77d"),
    "xgb_v1": ("89604e7c85501214698f308e0e8a4ad98af7acba84bae434761b7591808a6632", "eb3271a13ed415b819c400230f2f3f9d6e816076c256193139487775a83f0e48"),
    "xgb_v2": ("aa8dc9c0a642601f6aef61d2129f35ebb5cc806e5ffde8a1c6f29ba8112217cc", "ac8cb290430b427778b62f7ed328c51f2143355dadc2b2ab8b04ea558b15aa00"),
}.items():
    EXT_FILES[f"{n_} (mathsonchain)"] = (f"oof_{n_}.npy", o_, f"test_{n_}.npy", t_)
EXT_O, EXT_T = {}, {}
for n_, (fo, ho, ft, ht) in EXT_FILES.items():
    po, pt = find(fo, ho, required=False), find(ft, ht, required=False)
    if po is None or pt is None or sha256(po) != ho or sha256(pt) != ht: print("  skipped (not attached or changed):", n_); continue
    EXT_O[n_] = L(np.load(po, allow_pickle=False).astype(np.float64)); EXT_T[n_] = L(np.load(pt, allow_pickle=False).astype(np.float64))
EXT_NAMES = list(EXT_O)
WGRID = np.array([-0.05, -0.03, -0.02, -0.01, 0.01, 0.02, 0.03, 0.05])
print(f"{len(EXT_NAMES)} candidate members x {len(WGRID)} weights = {len(EXT_NAMES) * len(WGRID)} candidate steps")

def ext_round(B):
    """Per-fold AUC gain (x 1e-5) of every candidate step on base B, then the nested choice and the gate."""
    base_f = np.array([auc_sort(y[b], B[b]) for b in FOLDS])
    G = np.zeros((len(EXT_NAMES), len(WGRID), 5))
    for i, n_ in enumerate(EXT_NAMES):
        D = EXT_O[n_] - B
        for j, w in enumerate(WGRID):
            s = B + w * D
            G[i, j] = (np.array([auc_sort(y[b], s[b]) for b in FOLDS]) - base_f) * 1e5
    outer, picks = np.zeros(5), []
    for f in range(5):
        inner = [g for g in range(5) if g != f]
        i, j = np.unravel_index(G[:, :, inner].mean(2).argmax(), G.shape[:2])
        outer[f] = G[i, j, f]; picks.append((EXT_NAMES[i], WGRID[j]))
    i, j = np.unravel_index(G.mean(2).argmax(), G.shape[:2])
    passed = (outer > 0).sum() >= 4 and outer.mean() > 0
    return dict(G=G, outer=outer, picks=picks, final=(EXT_NAMES[i], WGRID[j]), final_in=G[i, j].mean(), passed=passed)

B_O, B_T = L(MO["388"]), logit(S388)
EXT, cur_O, cur_T = [], B_O, B_T
for step in (1, 2):
    rd = ext_round(cur_O); EXT.append(rd)
    print(f"step {step}: outer-fold gains {np.round(rd['outer'], 1)} (positive on {(rd['outer'] > 0).sum()}/5, mean {rd['outer'].mean():+.1f}); "
          f"inner picks {[f'{a} {b:+.2f}' for a, b in rd['picks']]}")
    print(f"        best step on all five folds (in-sample): {rd['final'][0]} {rd['final'][1]:+.2f}, {rd['final_in']:+.1f}e-5 -> gate {'PASSED' if rd['passed'] else 'NOT passed'}")
    if not rd["passed"]: break
    n_, w = rd["final"]
    cur_O = cur_O + w * (EXT_O[n_] - cur_O); cur_T = cur_T + w * (EXT_T[n_] - cur_T)
N_STEPS = sum(r["passed"] for r in EXT)
SUB = S388 if N_STEPS == 0 else (rankdata(cur_T) - 0.5) / NT
sample[["id"]].assign(is_fraud=SUB).to_csv(WORK / "submission.csv", index=False)
SUB_SHA = sha256(WORK / "submission.csv")
print(f"submission.csv: {N_STEPS} extension step(s); SHA-256 {SUB_SHA}; identical to the rebuilt 0.95388 file: {SUB_SHA == sha256(_chk)}")
if N_STEPS:
    print(f"  rank correlation with the 0.95388 file {np.corrcoef(rankdata(SUB), rankdata(S388))[0, 1]:.6f}; "
          f"public-gap noise of the change ~{sig(np.corrcoef(rankdata(SUB), rankdata(S388))[0, 1]):.1f}e-5")
del EXT_T

In [ ]:
# ERROR ANALYSIS on the OOF rows (the stand-in test set): the honest wide LM stack W, the 0.95370 replay REF, the bold mix
# and the 0.95388 file's OOF counterpart. Every change below is computed by applying a step to a SUBSET of rows only.
EA = {"wide LM stack": W, "0.95370 replay": REF_OOF, "bold mix": L(PO["bold"]), "0.95388 file": B_O}
EA = {k: L(v) for k, v in EA.items()}
EA_AUC = {k: roc_auc_score(y, v) for k, v in EA.items()}
for k, v in EA_AUC.items(): print(f"  {k:16s} OOF AUC {v:.6f}")
UR = {k: (rankdata(v) - 0.5) / N for k, v in EA.items()}
STEPS_EA = [("0.95370 replay", "bold mix"), ("bold mix", "0.95388 file"), ("wide LM stack", "0.95388 file")]
def masked_gain(a, b, m):
    """AUC change (x1e-5) when file a takes file b's logit ranks on the rows in mask m only."""
    s = EA[a].copy(); s[m] = EA[b][m]; return (auc_sort(y, s) - auc_sort(y, EA[a])) * 1e5
# (a) where the files disagree: the most-moved rows first
QGRID = [0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.50, 1.0]
DISAG = {}
for a, b in STEPS_EA:
    mv = np.abs(UR[b] - UR[a]); order_ = np.argsort(-mv)
    gains = []
    for q in QGRID:
        m = np.zeros(N, bool); m[order_[:int(round(q * N))]] = True; gains.append(masked_gain(a, b, m))
    DISAG[(a, b)] = dict(gain=np.array(gains), fraud_top1=y[order_[:N // 100]].mean(), mv=mv)
    print(f"  {a} -> {b}: full change {gains[-1]:+.1f}e-5; the 1% most-moved rows alone {gains[1]:+.1f}, 5% {gains[3]:+.1f}; fraud rate in that 1% {y[order_[:N // 100]].mean():.1%}")
# (b) segments: score band of the base file, how common the transaction amount is (value count over train + test), channel
_amt_te = pd.read_csv(find("test.csv", inside=COMP), usecols=["amount_usd"])["amount_usd"]
_vc = pd.concat([train["amount_usd"], _amt_te]).value_counts()
AMT_CNT = train["amount_usd"].map(_vc).to_numpy()
SEGS = {
    "score band (rank in the base file)": lambda base: [("bottom 80%", UR[base] < 0.80), ("80-95%", (UR[base] >= 0.80) & (UR[base] < 0.95)),
                                                        ("95-99%", (UR[base] >= 0.95) & (UR[base] < 0.99)), ("top 1%", UR[base] >= 0.99)],
    "how common the amount is": lambda base: [("unique", AMT_CNT == 1), ("2-3 rows", (AMT_CNT >= 2) & (AMT_CNT <= 3)),
                                              ("4-20 rows", (AMT_CNT >= 4) & (AMT_CNT <= 20)), ("> 20 rows", AMT_CNT > 20)],
    "channel": lambda base: [(str(c), (train["channel"] == c).to_numpy()) for c in sorted(train["channel"].unique())],
}
SEG_TAB = []
for a, b in STEPS_EA:
    for sname, fn in SEGS.items():
        for lab, m in fn(a):
            SEG_TAB.append(dict(step=f"{a} -> {b}", segmentation=sname, segment=lab, rows=m.mean(), frauds=y[m].sum() / y.sum(),
                                gain=masked_gain(a, b, m), moved=np.abs(UR[b] - UR[a])[m].mean() * 1e3,
                                lift=((UR[b] - UR[a])[m & y].mean() - (UR[b] - UR[a])[m & ~y].mean()) * 1e3))
    for lab, m in [("frauds only", y), ("legit rows only", ~y), ("frauds ranked below 80%", y & (UR[a] < 0.80))]:
        SEG_TAB.append(dict(step=f"{a} -> {b}", segmentation="by label (analysis only)", segment=lab, rows=m.mean(), frauds=y[m].sum() / y.sum(),
                            gain=masked_gain(a, b, m), moved=np.abs(UR[b] - UR[a])[m].mean() * 1e3, lift=np.nan))
SEG_TAB = pd.DataFrame(SEG_TAB)
# (c) rank shift of frauds and legit rows along the base file's score (calibration of the order)
EDGES = np.r_[0, 0.5, 0.8, 0.9, 0.95, 0.97, 0.98, 0.99, 0.995, 0.998, 1.0]
SHIFT = {}
for a, b in STEPS_EA:
    d = UR[b] - UR[a]; band = np.clip(np.searchsorted(EDGES, UR[a], side="right") - 1, 0, len(EDGES) - 2)
    SHIFT[(a, b)] = np.array([[d[(band == i) & y].mean() * 1e3, d[(band == i) & ~y].mean() * 1e3, (band == i).sum(), (y & (band == i)).sum()]
                              for i in range(len(EDGES) - 1)])
CAPTURE = {k: [y[UR[k] >= 1 - q].sum() / y.sum() for q in (0.01, 0.02, 0.05)] for k in EA}
log("error analysis done")

<a id="hero"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">The whole story in one figure</div>

In [ ]:
# Hero figure: KPI tiles + four panels (rungs vs noise, public vs nested CV, winner's curse, the top band).
PUB_FRAUD = NPUB * y.mean()
MV = {m[0]: m for m in MOVES}
VARIANT = {"GPT-2 seed swap 0.10"}                               # the same rung at another weight: counted once
POS = [m[0] for m in MOVES if m[2] > 0 and m[0] not in VARIANT]  # distinct probes the public board rewarded
CONF = [n for n in POS if TRUE[n] > 0 and UP[n] >= 4]            # ... that nested CV also confirms (>= 4 of 5 folds)
SHORT = {"+0.12 megayak LM": "+ megayak LM", "+0.06 seed-7 LM": "+ seed-7 LM", "+0.12 public GPT-2": "+ public GPT-2",
         "+0.05 GP + DVAE": "+ GP + DVAE", "LM weight 0.12 -> 0.09": "LM 0.12 -> 0.09", "more of both LMs": "more of both LMs",
         "less seed-7 LM": "less seed-7 LM", "GPT-2 seed swap 0.06": "GPT-2 seed swap", "GPT-2 seed swap 0.10": "seed swap at 0.10"}

def hero_rungs(ax):
    steps = STEPS if HAVE_LADDER else [(n, MV[n][2], ONE_RHO[NAMES.index(n)], SD_PUB[NAMES.index(n)]) for n in POS]
    yy = np.arange(len(steps))[::-1]
    for yi, (lab, g, _, sd) in zip(yy, steps):
        ax.add_patch(FancyBboxPatch((-2 * sd, yi - 0.34), 4 * sd, 0.68, boxstyle="round,pad=0,rounding_size=0.12", facecolor=BAND, edgecolor="none", zorder=1))
        ax.add_patch(FancyBboxPatch((-sd, yi - 0.34), 2 * sd, 0.68, boxstyle="round,pad=0,rounding_size=0.12", facecolor="#d6d5d0", edgecolor="none", zorder=1))
        ax.barh(yi, g, height=0.36, color=PUB, zorder=3)
        ax.annotate(f"+{g}", (max(g, 2 * sd), yi), xytext=(6, 0), textcoords="offset points", va="center", fontsize=10, fontweight="bold", color=INK)
        ax.annotate(f"noise SD {sd:.1f}", (max(g, 2 * sd), yi), xytext=(30, 0), textcoords="offset points", va="center", fontsize=8.5, color=INK2)
    ax.set_yticks(yy); ax.set_yticklabels([s[0] for s in steps], fontsize=9); ax.tick_params(axis="y", length=0)
    lim = max(max(s[1] for s in steps), 2 * max(s[3] for s in steps))
    ax.set_xlim(-lim * 1.05, lim * 2.0); ax.set_ylim(-0.6, len(steps) - 0.4)
    ax.xaxis.set_major_formatter(E5); ax.grid(axis="y", visible=False)
    ax.set_xlabel("public LB change of the step (x 1e-5)")
    zero_line(ax, "x")
    ax.legend(handles=[Patch(color=PUB, label="reported public gain"), Patch(color="#d6d5d0", label="1 SD of public noise"),
                       Patch(color=BAND, label="2 SD")], loc="upper center", bbox_to_anchor=(0.5, -0.17), ncol=3, fontsize=8.5)

def hero_scatter(ax):
    moves = [m[0] for m in MOVES]
    for n in moves:
        i = NAMES.index(n); x, yv = TRUE[n], MV[n][2]; good = n in CONF or (n in VARIANT and TRUE[n] > 0 and UP[n] >= 4)
        ax.plot([x, x], [yv - SD_PUB[i], yv + SD_PUB[i]], color=RULE, lw=2.2, solid_capstyle="round", zorder=2)
        ax.scatter(x, yv, s=110 if good else 60, color=CV if good else PUB, edgecolor=SURF, lw=1.5, zorder=4)
    ax.set_xlim(-9.5, 4); ax.set_ylim(-16.5, 6)
    ax.plot([-9.5, 4], [-9.5, 4], color=MUTED, lw=1, ls=(0, (4, 3)), zorder=1)
    ax.text(-9.3, -10.0, "dashed: public = CV", color=MUTED, fontsize=8.5, va="top")
    zero_line(ax, "x"); zero_line(ax, "y")
    ax.annotate("+ public GPT-2: -14 public,\nand down on 5 of 5 folds", (TRUE["+0.12 public GPT-2"], -14), xytext=(14, 0), textcoords="offset points",
                fontsize=8.5, color=INK2, va="center")
    ax.annotate("+ megayak LM", (TRUE["+0.12 megayak LM"], MV["+0.12 megayak LM"][2]), xytext=(-8, 9), textcoords="offset points",
                fontsize=8.5, color=INK2, ha="right")
    ax.annotate("GPT-2 seed swap:\nup on both boards", (TRUE["GPT-2 seed swap 0.06"], 2), xytext=(4, 34), textcoords="offset points",
                fontsize=9, color=CVTXT, fontweight="bold", ha="center", arrowprops=dict(arrowstyle="-", color=CVTXT, lw=1))
    cl = [n for n in moves if n not in ("+0.12 public GPT-2", "+0.12 megayak LM", "GPT-2 seed swap 0.06", "GPT-2 seed swap 0.10")]
    x0, x1 = min(TRUE[n] for n in cl) - 0.35, max(TRUE[n] for n in cl) + 0.35
    y0, y1 = min(MV[n][2] for n in cl) - 0.8, max(MV[n][2] for n in cl) + 0.8
    ax.add_patch(FancyBboxPatch((x0, y0), x1 - x0, y1 - y0, boxstyle="round,pad=0,rounding_size=0.4", facecolor="none",
                                edgecolor=PUB, lw=1, ls=(0, (2, 2)), zorder=3))
    ax.annotate(f"{len(cl)} more probes: public {min(MV[n][2] for n in cl):+d} to {max(MV[n][2] for n in cl):+d},\n"
                f"nested CV {min(TRUE[n] for n in cl):+.1f} to {max(TRUE[n] for n in cl):+.1f}", (x0, y0 + 0.2), xytext=(-10, -26),
                textcoords="offset points", fontsize=8.5, color=INK2, ha="right", va="top", arrowprops=dict(arrowstyle="-", color=PUB, lw=0.8))
    ax.xaxis.set_major_formatter(E5); ax.yaxis.set_major_formatter(E5)
    ax.set_xlabel("change in nested CV on the public OOF (x 1e-5)"); ax.set_ylabel("reported public LB change (x 1e-5)")
    ax.text(0.99, 0.02, "grey bars: 1 SD of public noise", transform=ax.transAxes, ha="right", va="bottom", fontsize=8, color=MUTED)

def hero_curse(ax):
    ax.plot(KS, SIG_RUNG * EMAX, color=PUB, lw=2, label="best public gain of K null probes (theory)")
    ax.fill_between(KS, 0, SIG_RUNG * EMAX, color=PUB, alpha=0.07, lw=0)
    kk = np.array(KGRID)
    ax.plot(kk, [CURSE[k][:, 0].mean() for k in KGRID], "o", color=PUB, ms=7, mec=SURF, mew=1.5, label="best of K real probes: public part")
    ax.plot(kk, [CURSE[k][:, 1].mean() for k in KGRID], "o-", color=PRIV, ms=7, mec=SURF, mew=1.5, lw=1.4, label="the same pick: private part")
    zero_line(ax)
    ax.set_ylim(min(-4, min(CURSE[k][:, 1].mean() for k in KGRID) - 2), SIG_RUNG * EMAX[-1] * 1.12)
    med = int(np.median(LB14_SUBS[:50]))
    ax.axvline(med, color=MUTED, lw=1, ls=(0, (3, 3)))
    ax.text(med * 1.07, SIG_RUNG * EMAX[-1] * 0.45, f"median top-50 team:\n{med} submissions", fontsize=8.5, color=INK2, va="top")
    ax.set_xscale("log"); ax.set_xticks([1, 2, 5, 10, 20, 50, 100]); ax.set_xticklabels(["1", "2", "5", "10", "20", "50", "100"])
    ax.yaxis.set_major_formatter(E5); ax.yaxis.set_major_locator(MaxNLocator(6, integer=True))
    ax.set_xlabel("number of probes K compared on the public board")
    ax.set_ylabel("score change of the chosen probe (x 1e-5)"); ax.legend(loc="upper left", fontsize=8.5)

def hero_board(ax):
    lo, hi, ref = 95330, 95450, 388                      # 14 Nov snapshot; #1 and #2 lie far to the right
    v14 = LB14_TOP[(LB14_TOP + 95000 >= lo) & (LB14_TOP + 95000 <= hi)]
    vals, cnt = np.unique(v14, return_counts=True)
    ax.axvspan(ref - 2 * SIG_RUNG, ref + 2 * SIG_RUNG, color=BAND, zorder=0, lw=0)
    ax.text(ref - 2 * SIG_RUNG - 0.8, cnt.max() * 1.22, f"within 2 rung-noise SDs\nof 0.95388: {N_2SIG14} teams", fontsize=8.5, color=INK2, va="top", ha="right")
    ax.bar(vals, cnt, width=0.75, color=[PUB if c < 10 else NAVY for c in cnt], zorder=3)
    for v, c in zip(vals, cnt):
        if c >= 15: ax.text(v, c + 0.6, f"0.95{v:03d}\n{c} teams", ha="center", va="bottom", fontsize=8, color=INK)
    ax.annotate(f"#2 0.95{LB14_TOP[1]}, #1 0.95{LB14_TOP[0]}\n(far off this scale)", xy=(hi - 95000, cnt.max() * 0.25), xytext=(-8, 0),
                textcoords="offset points", ha="right", va="center", fontsize=8.5, color=PRIV, fontweight="bold",
                arrowprops=None)
    ax.annotate("", xy=(hi - 95000 + 0.6, cnt.max() * 0.25), xytext=(hi - 95000 - 0.2, cnt.max() * 0.25),
                arrowprops=dict(arrowstyle="->", color=PRIV, lw=1.4))
    ax.set_xlim(lo - 95000 - 1, hi - 95000 + 1); ax.set_ylim(0, cnt.max() * 1.3)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"0.95{int(v):03d}")); ax.xaxis.set_major_locator(MaxNLocator(7, integer=True))
    ax.set_xlabel("public LB score (14 Nov)"); ax.set_ylabel("teams with exactly this score")

N_2SIG14 = int((np.abs(LB14_TOP - 388) <= 2 * SIG_RUNG).sum())
SD388 = sig(rho_te("bold + 0.5 (bold - balanced)", "bold"))
fig = plt.figure(figsize=(16, 12.4))
fig.text(0.02, 0.975, "How much of the public top is real?", fontsize=24, fontweight="bold", color=INK, va="top")
fig.text(0.02, 0.937, "Ladder steps of one or two points sit inside the noise of the 60k-row public board, and nested CV confirms few of them. "
         "New in v4: the 0.95388 ladder, checked the same way.", fontsize=12, color=INK2, va="top")
tw, gap, ty = 0.2275, 0.0133, 0.815
TILES = [("Noise of one ladder rung", f"±{SIG_RUNG:.1f}e-5", "1 SD of the public gap between consecutive rungs", NAVY),
         ("Old-ladder gains confirmed by nested CV", f"{len(CONF)} of {len(POS)}", "public probes that also gain on >= 4 of 5 folds", CV),
         ("New: the step to 0.95388", f"+2 vs ±{SD388:.1f}", "public gain vs 1 SD of its noise; best of 8 probes", PUB),
         ("New: honest steps on top of 0.95388", f"{N_STEPS} of {len(EXT_NAMES) * len(WGRID)}", "candidate steps that pass the nested 4-of-5 gate", PRIV)]
for i, t_ in enumerate(TILES): tile(fig, [0.02 + i * (tw + gap), ty, tw, 0.095], *t_)
axs = [fig.add_axes([0.205, 0.465, 0.29, 0.26]), fig.add_axes([0.605, 0.43, 0.37, 0.295]),
       fig.add_axes([0.06, 0.06, 0.40, 0.28]), fig.add_axes([0.56, 0.06, 0.41, 0.28])]
hero_rungs(axs[0]);   panel_title(axs[0], "A  Each rung gains about one noise SD", "public gain of consecutive ladder files vs. the noise of that gap")
hero_scatter(axs[1]); panel_title(axs[1], "B  Public says up; nested CV mostly says flat or down", "every public probe with OOF files behind it, replayed on the wide-stack OOF")
hero_curse(axs[2]);   panel_title(axs[2], "C  The best of many probes gains on public, not on private", "random public/private draws from the OOF; probes = small steps towards public members")
hero_board(axs[3]);   panel_title(axs[3], "D  The top band is a few rung-noise SDs wide", "public leaderboard, 14 Nov; tall bars = many teams at exactly the score of a public notebook")
plt.show()

<div style="background: #eef4fc; border-left: 5px solid #2a78d6; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>In one paragraph.</b> A ladder rung of one or two points is about one noise SD of the public gap between the two
files (panel A). Replayed on a nested OOF, most rungs that won on public are flat or negative; the one that holds up,
a second seed of the same model, has an obvious mechanism (panel B). Choosing the best of many probes by public score
guarantees a public gain even when nothing is better, and the private part does not repeat it (panel C). The top of
the board is only a few such noise SDs wide, and many teams sit on the exact score of a public notebook (panel D).
</div>

<a id="s1"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">1 &middot; The competition in five minutes</div>

* **Data.** 699,960 synthetic card transactions to train on, 299,984 to predict; 24 columns (amount, timing, device,
  merchant, risk scores, flags). About **1.67%** of the training rows are fraud.
* **Metric.** ROC-AUC: the probability that a random fraud is ranked above a random legitimate transaction. Only the
  **order** of the predictions matters, which is why every public blend here works on logit ranks
  `L(v) = logit((rank(v) - 0.5) / n)`.
* **The split.** The public leaderboard scores **20%** of the test rows, the private leaderboard (the final ranking)
  the other **80%**. Both are fixed for everyone; nobody knows which rows are which.
* **Units.** Gaps at the top are a few hundred-thousandths of AUC. We write them as **points of 1e-5**:
  0.95354 - 0.95352 = +2 points.

In [ ]:
# The two boards in rows and expected frauds (train fraud rate applied to the test set).
fig, ax = plt.subplots(figsize=(13, 2.9))
rate = y.mean()
segs = [("train", 0, N, MUTED, f"train  {N:,} rows  |  {int(y.sum()):,} frauds"),
        ("public", 0, NPUB, PUB, f"public LB  ~{NPUB:,} rows\n~{NPUB * rate:,.0f} frauds"),
        ("private", NPUB, NT - NPUB, PRIV, f"private LB  ~{NT - NPUB:,} rows  |  ~{(NT - NPUB) * rate:,.0f} frauds")]
ax.barh(1, N, color="#e6e5df", height=0.55); ax.text(N / 2, 1, segs[0][4], ha="center", va="center", fontsize=10.5, color=INK)
ax.barh(0, NPUB, color=PUB, height=0.55); ax.barh(0, NT - NPUB, left=NPUB + 1500, color=PRIV, height=0.55)
ax.text(NPUB / 2, 0, segs[1][4], ha="center", va="center", fontsize=9.5, color="white", fontweight="bold")
ax.text(NPUB + (NT - NPUB) / 2, 0, segs[2][4], ha="center", va="center", fontsize=10.5, color="white", fontweight="bold")
ax.set_yticks([1, 0]); ax.set_yticklabels(["train", "test"], fontsize=11); ax.tick_params(axis="y", length=0)
ax.set_xlim(0, N * 1.01); ax.grid(False); ax.spines["left"].set_visible(False)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1000:.0f}k")); ax.set_xlabel("rows")
headline(fig, f"The public board ranks everyone on about {NPUB * rate:,.0f} frauds",
         "Public = 20% of the test rows, private = the other 80%. Frauds are estimated from the training fraud rate.")
plt.show()

<a id="s2"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">2 &middot; How noisy is the public LB?</div>

Two different questions hide behind "the public LB is noisy":

* **How far is one file's public score from its private score?** Far: the AUC of a fixed file on ~60k rows with about
  a thousand frauds wanders by a few *thousandths*. But every team is scored on the same rows, so this mostly shifts
  the whole board together.
* **How reliable is the public gap between two files?** This is what decides rankings and ladder steps. Two
  files that rank almost every row the same way share most of their noise, so the gap is far less noisy &mdash; but
  the gaps at the top are also tiny.

We measure both on the nested OOF of the public wide LM stack: draw a public part (~60k rows) and a private part
(~240k rows) at random, score every file on both, repeat a few hundred times.

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.4))
gap1 = (AUC[0, :, 0] - AUC[0, :, 1]) * 1e5
ax.hist(gap1, bins=30, color=MUTED, edgecolor=SURF)
ax.set_xlabel("public AUC minus private AUC of ONE file (x 1e-5)"); ax.set_ylabel("random splits"); ax.xaxis.set_major_formatter(E5)
panel_title(ax, f"One file: SD {np.std(gap1):.0f} points", "the whole board shares this shift")
iS = NAMES.index("GPT-2 seed swap 0.06")
bins_ = np.linspace(min(DELTA[iS, :, 0].min(), DELTA[iS, :, 1].min()), max(DELTA[iS, :, 0].max(), DELTA[iS, :, 1].max()), 36)
ax2.hist(DELTA[iS, :, 0], bins=bins_, color=PUB, edgecolor=SURF, alpha=0.9, label="public gap")
ax2.hist(DELTA[iS, :, 1], bins=bins_, color=PRIV, edgecolor=SURF, alpha=0.75, label="private gap")
ax2.axvline(TRUE["GPT-2 seed swap 0.06"], color=INK, lw=1.5)
ax2.text(TRUE["GPT-2 seed swap 0.06"] + 0.3, ax2.get_ylim()[1] * 0.95, f"nested CV {TRUE['GPT-2 seed swap 0.06']:+.1f}", fontsize=9, va="top")
zero_line(ax2, "x"); ax2.xaxis.set_major_formatter(E5); ax2.legend(loc="upper left", fontsize=9)
ax2.set_xlabel("gap between two consecutive ladder files (x 1e-5)")
panel_title(ax2, f"Two near-identical files: public SD {SD_PUB[iS]:.1f}, private SD {SD_PRV[iS]:.1f}",
            f"the GPT-2 seed swap: public gap < 0 in {np.mean(DELTA[iS, :, 0] < 0):.0%} of splits")
headline(fig, f"One public score swings by +-{np.std(gap1):.0f} points; the gap between two near-identical files by +-{SD_PUB[iS]:.1f}, still more than a rung",
         "Independent random public (~60k rows) and private (~240k rows) draws from the public wide-stack OOF")
plt.show()

### A one-line rule for the noise of a public gap
For two files `a` and `b`, the noise of their public gap depends almost only on **how differently they rank the rows**,
that is on `1 - rho`, where `rho` is the rank (Spearman) correlation of the two files. Over every move measured here the
relation is a clean square root:

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.2))
order_ = np.argsort(ONE_RHO)
is_move = np.array([n in MV or n == "tree move" for n in NAMES])
ax.scatter(ONE_RHO[~is_move], SD_PUB[~is_move], s=36, color=PUB, alpha=0.55, edgecolor="none", label="random probe steps (public part)")
ax.scatter(ONE_RHO[is_move], SD_PUB[is_move], s=60, color=PUB, edgecolor=SURF, lw=1.2, label="public ladder probes (public part)")
ax.scatter(ONE_RHO, SD_PRV, s=22, color=PRIV, alpha=0.6, edgecolor="none", label="the same steps, private part")
xx = np.logspace(np.log10(ONE_RHO.min() * 0.6), np.log10(ONE_RHO.max() * 1.6), 50)
ax.plot(xx, K_RULE * np.sqrt(xx) * 1e5, color=NAVY, lw=2)
ax.plot(xx, K_RULE * np.sqrt(xx) * 1e5 * np.sqrt(NPUB / (NT - NPUB)), color=PRIV, lw=1.5, ls=(0, (4, 3)))
ax.text(xx[-1], K_RULE * np.sqrt(xx[-1]) * 1e5, f"  public: {K_RULE:.4f} x sqrt(1 - rho)", va="center", fontsize=10, fontweight="bold", color=NAVY)
ax.text(xx[-1], K_RULE * np.sqrt(xx[-1]) * 1e5 * 0.5, "  private: half of it", va="center", fontsize=9.5, color=PRIV)
if HAVE_LADDER:
    for lab, g, orho, sd in STEPS:
        ax.axvline(orho, color=RULE, lw=1, zorder=0)
    ax.text(max(s[2] for s in STEPS) * 1.08, ax.get_ylim()[0] * 1.15, "grey lines: consecutive\npublic ladder files", fontsize=8.5, color=INK2, va="bottom")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("1 - rank correlation of the two files"); ax.set_ylabel("SD of the gap (x 1e-5)")
ax.set_xlim(xx[0], xx[-1] * 4.5); ax.legend(loc="upper left", fontsize=9)
headline(fig, f"Noise of a public gap = {K_RULE:.4f} x sqrt(1 - rho): about {K_RULE * np.sqrt(1e-4) * 1e5:.0f} points when rho = 0.9999",
         "Each dot: SD of the gap between the wide-stack OOF and one small move of it, over random 20/80 splits")
plt.show()

In [ ]:
# Shown: use the rule on your own two submission files (the constant K_RULE was fitted above on public OOF; it assumes
# a test set and fraud rate like this competition's, and two files of similar strength).
def public_gap_noise(file_a, file_b, k=K_RULE):
    """1 SD (in AUC) of the public-LB gap between two submissions, from their rank correlation alone."""
    a = pd.read_csv(file_a)["is_fraud"].to_numpy(); b = pd.read_csv(file_b)["is_fraud"].to_numpy()
    rho = np.corrcoef(rankdata(a), rankdata(b))[0, 1]
    return k * np.sqrt(max(1 - rho, 0)), rho
for r_ in (0.99999, 0.9999, 0.9995, 0.999, 0.995):
    print(f"rho {r_:<8}  public-gap noise ~ {K_RULE * np.sqrt(1 - r_) * 1e5:5.1f} points   private ~ {K_RULE * np.sqrt(1 - r_) * 1e5 / 2:4.1f}")

<div style="background: #eef4fc; border-left: 5px solid #2a78d6; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>How to read it.</b> Two files with rank correlation 0.9999 have a public gap noise of a few points, about the size
of a typical ladder step. A gain of one or two points between such files is <b>not evidence</b> of anything: the same
change would show a loss on public in a large share of possible splits. The private gap is half as noisy (four times
more rows), so the private board will not simply repeat the public order.
</div>

<a id="s3"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">3 &middot; Anatomy of a ladder</div>

A ladder starts from a strong public file and adds small steps in logit-rank space, keeping each step that the public
score rewards. The current public top (0.95354) is built like this, and its authors published **every rung and every
probe** (goodpjw2008's *LM Ladder*, Areeb Ali's *Three Ladders*):

```
s352  = A + 0.6 (A - F) + 0.09 (LM - F) + 0.09 (L7F - F) + 0.05 (GPD - F) - 0.05 (MLP - F) - 0.025 (GB - F)   # 0.95352
score = L(s352) + 0.06 (G26 - MS)                                                                            # 0.95354
```

`A` = Strike 10 (amanatar, 0.95340), `F` = Strike 9 (Moran Z, 0.95330), `LM` = megayak's target-first LM, `L7F` =
goodpjw2008's seed-7 LM, `GPD` = daoviet's GP + DVAE, `MLP`/`GB` = two OOF-bank bags, `MS` = masayakawamata's
target-first GPT-2, `G26` = a second seed of it (goodpjw2008). The notebook rebuilt this file above from the public
inputs and checked it byte for byte against the published one.

**The part no one can check.** `A` and `F` publish no OOF predictions, so the base and the extrapolation
`0.6 (A - F)` (move further along the direction from Strike 9 to Strike 10) have no out-of-fold check at all. Every
other ingredient has public OOF files, so each step can be replayed on a nested OOF, as the ladder authors did
themselves. The table collects all of them.

In [ ]:
rows = []
for n, d, pub, src in MOVES:
    i = NAMES.index(n)
    rows.append({"probe (step applied to the ladder file)": n, "public LB change": f"{pub:+d}",
                 "nested CV change": f"{TRUE[n]:+.1f}", "folds up": f"{UP[n]}/5",
                 "public noise SD": f"{SD_PUB[i]:.1f}", "P(public sign = private sign)": f"{AGREE[i]:.0%}", "source of the public number": src})
i = NAMES.index("tree move")
rows.append({"probe (step applied to the ladder file)": "full tree move (-0.10 MLP, -0.05 GB)", "public LB change": "part of v15",
             "nested CV change": f"{TRUE['tree move']:+.1f}", "folds up": f"{UP['tree move']}/5", "public noise SD": f"{SD_PUB[i]:.1f}",
             "P(public sign = private sign)": f"{AGREE[i]:.0%}", "source of the public number": "Areeb Ali v15 (with other changes)"})
TAB = pd.DataFrame(rows)
print("All changes in points of 1e-5. Nested CV = applied to the nested OOF of the public wide LM stack (CV %.6f)." % CV_W)
TAB.style.hide(axis="index") if hasattr(TAB.style, "hide") else TAB

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.6))
names_ = [m[0] for m in MOVES] + ["tree move"]
pubv = [m[2] for m in MOVES] + [np.nan]
cvv = [TRUE[n] for n in names_]
sdv = [SD_PUB[NAMES.index(n)] for n in names_]
yy = np.arange(len(names_))[::-1]
for yi, s in zip(yy, sdv):
    ax.add_patch(FancyBboxPatch((-s, yi - 0.38), 2 * s, 0.76, boxstyle="round,pad=0,rounding_size=0.1", facecolor=BAND, edgecolor="none", zorder=0))
ax.barh(yy + 0.17, pubv, height=0.32, color=PUB, label="reported public LB change", zorder=3)
ax.barh(yy - 0.17, cvv, height=0.32, color=[CV if (c > 0 and UP[n] >= 4) else "#9ccfba" if c > 0 else PRIV for c, n in zip(cvv, names_)],
        label="nested CV change", zorder=3)
for yi, n, c in zip(yy, names_, cvv):
    ax.text(max(0, c) + 0.4 if c >= 0 else c - 0.4, yi - 0.17, f"{c:+.1f}  ({UP[n]}/5 folds)", va="center",
            ha="left" if c >= 0 else "right", fontsize=8.5, color=INK2)
ax.set_yticks(yy); ax.set_yticklabels([SHORT.get(n, "full tree move") for n in names_], fontsize=9.5); ax.tick_params(axis="y", length=0)
zero_line(ax, "x"); ax.xaxis.set_major_formatter(E5); ax.set_xlabel("change (x 1e-5)"); ax.grid(axis="y", visible=False)
ax.set_xlim(-17, 8)
ax.legend(handles=[Patch(color=PUB, label="reported public LB change"), Patch(color=CV, label="nested CV: up on >= 4 of 5 folds"),
                   Patch(color="#9ccfba", label="nested CV: up on < 4 folds"), Patch(color=PRIV, label="nested CV: down"), Patch(color=BAND, label="+-1 SD public noise of the step")],
          loc="lower left", fontsize=9)
headline(fig, f"Only the GPT-2 seed swap gains on both boards; {len(POS) - len(CONF)} other public gains are not confirmed by nested CV",
         "Every public probe that has OOF files behind it, replayed on the nested OOF of the public wide LM stack")
plt.show()

**Reading the table.** The ladder authors saw the same thing and said so: Areeb Ali writes that *"every earlier ladder
step lost on CV"* and that the two-seed rung is the first one that holds up; goodpjw2008 calls a rung *"a bet, not a
measurement"*. Our replay agrees with their own CV checks: the two-seed rung gains about +1.9e-5 on 5 of 5 folds
(goodpjw2008: +1.9e-5) and the full tree move loses about 7.7e-5 (Areeb Ali: -7.4e-5).

The two-seed rung is also the one with a clear mechanism: averaging two seeds of the same model removes part of one
seed's noise, which is a gain one expects on any rows. The other steps tune *how much* of a member the file holds, and
there the public board and CV disagree; with a public noise SD of 3&ndash;7 points per step, the public board cannot
settle a one- or two-point question.

<a id="s4"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">4 &middot; The winner's curse</div>

Try `K` probes, keep the one with the best public score. Even if none of them is truly better, the best of `K` noisy
numbers is above zero, by about `sigma x E[max of K standard normals]`: about 0.8 sigma for K = 3, 1.5 sigma for K = 10,
2.0 sigma for K = 30, 2.5 sigma for K = 100. The private part of the same split does not share that luck, so the pick
falls back. This is the textbook *winner's curse* (selection bias) and, repeated over many rounds of public feedback,
*adaptive overfitting* of the public holdout.

Below, the probes are real: small steps from the wide stack towards (or away from) each OOF-bank member and each public
LM. On every random split we pick the best of `K` by its public gap and read its private gap.

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw=dict(width_ratios=[1.15, 1]))
kk = np.array(KGRID)
pub_m = np.array([CURSE[k][:, 0].mean() for k in KGRID]); prv_m = np.array([CURSE[k][:, 1].mean() for k in KGRID])
tru_m = np.array([CURSE[k][:, 2].mean() for k in KGRID])
ax.fill_between(kk, prv_m, pub_m, color=PUB, alpha=0.08, lw=0)
ax.plot(kk, pub_m, "o-", color=PUB, ms=7, mec=SURF, label="public gap of the pick")
ax.plot(kk, prv_m, "o-", color=PRIV, ms=7, mec=SURF, label="private gap of the same pick")
ax.plot(kk, tru_m, "o--", color=CV, ms=6, mec=SURF, lw=1.4, label="nested CV of the pick (its true size)")
ax.annotate(f"{pub_m[-1] - prv_m[-1]:.1f} points\nlost on private", xy=(kk[-1], (pub_m[-1] + prv_m[-1]) / 2), xytext=(-12, 0),
            textcoords="offset points", ha="right", va="center", fontsize=9.5, fontweight="bold", color=INK)
zero_line(ax); ax.set_xscale("log"); ax.set_xticks(KGRID); ax.set_xticklabels([str(k) for k in KGRID])
ax.yaxis.set_major_formatter(E5); ax.set_xlabel("probes compared on public (K)"); ax.set_ylabel("gap vs. the base file (x 1e-5)")
ax.legend(loc="upper left", fontsize=9)
panel_title(ax, "The more probes you compare, the more the pick is luck", "average over random 20/80 splits")
c20 = CURSE[20]
ax2.scatter(c20[:, 0], c20[:, 1], s=8, color=PUB, alpha=0.25, edgecolor="none")
lim = np.percentile(np.abs(c20[:, :2]), 99.5)
ax2.plot([-lim, lim], [-lim, lim], color=MUTED, lw=1, ls=(0, (4, 3)))
zero_line(ax2); zero_line(ax2, "x"); ax2.set_xlim(-lim, lim); ax2.set_ylim(-lim, lim)
ax2.xaxis.set_major_formatter(E5); ax2.yaxis.set_major_formatter(E5)
ax2.set_xlabel("public gap of the best of 20 (x 1e-5)"); ax2.set_ylabel("its private gap (x 1e-5)")
ax2.text(0.97, 0.05, f"public > 0 in {np.mean(c20[:, 0] > 0):.0%} of splits\nprivate > 0 in {np.mean(c20[:, 1] > 0):.0%}",
         transform=ax2.transAxes, ha="right", va="bottom", fontsize=9.5, color=INK)
panel_title(ax2, "Best of 20: almost always up on public", "each dot = one random split")
headline(fig, "Keeping the best public probe buys public points that the private board does not pay",
         "Real probes on the public wide-stack OOF; public and private parts drawn independently in each split")
plt.show()

<div style="background: #fff4ec; border-left: 5px solid #eb6834; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>How many probes are behind a public file?</b> The version tables list ten versions for one ladder (v9&ndash;v20) and
about twenty scored probes for the other, and these ladders build on each other. On the leaderboard snapshot the
median top-50 team had made about 20 submissions on 12 Nov (28 on 14 Nov), and the top teams over 100. With a rung noise of a few points,
the theory says the best of 20 null probes shows about 1.9 noise SDs of public "gain". That is the size of the whole
visible spread between many of the top files.
</div>

<a id="s5"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">5 &middot; CV can be tuned too</div>

Nested CV is the referee in this notebook, so it is fair to ask how a CV number can itself be inflated. Three common
ways, each measured below on public data:

1. **Early stopping on the scored fold.** The fold that reports the score also picks the number of trees.
2. **Selecting on the reported folds.** A blend, a member set or a weight chosen on the same OOF rows it is scored on.
3. **A test inside one fold.** An inner CV run on the rows of a single outer fold measures something else.

In [ ]:
# 5.1  Early stopping on the scored fold: one 4-leaf LightGBM per outer fold on the raw columns (learning rate 0.1),
# trained on 90% of the training part. We record the AUC every 10 trees on the scored fold AND on the held-out 10%,
# so the two ways of choosing the number of trees are compared on the very same model.
import lightgbm as lgb
NTH = int(os.environ.get("OMP_NUM_THREADS", 4))
X = train.drop(columns=["id", "is_fraud"]).copy()
for c in X.columns:
    if X[c].dtype == object: X[c] = X[c].astype("category")
    elif X[c].dtype == bool: X[c] = X[c].astype(int)
ES_FOLDS, ES_ROUNDS, STEP = [0, 1, 2], 1500, 10
ES = []
for k in ES_FOLDS:
    va = FOLDS[k]; tr = np.setdiff1d(np.arange(N), va)
    inner = np.random.default_rng(k).random(len(tr)) < 0.10
    a, h = tr[~inner], tr[inner]
    bst = lgb.train(dict(objective="binary", num_leaves=4, learning_rate=0.1, min_data_in_leaf=100, verbose=-1,
                         num_threads=NTH, seed=k), lgb.Dataset(X.iloc[a], y[a]), num_boost_round=ES_ROUNDS)
    zh, zv = np.zeros(len(h)), np.zeros(len(va)); its, ah, av = [], [], []
    for s in range(0, ES_ROUNDS, STEP):
        zh += bst.predict(X.iloc[h], start_iteration=s, num_iteration=STEP, raw_score=True)
        zv += bst.predict(X.iloc[va], start_iteration=s, num_iteration=STEP, raw_score=True)
        its.append(s + STEP); ah.append(roc_auc_score(y[h], zh)); av.append(roc_auc_score(y[va], zv))
    its, ah, av = map(np.array, (its, ah, av))
    i_sc, i_in = av.argmax(), ah.argmax()
    ES.append(dict(fold=k, its=its, av=av, i_sc=i_sc, i_in=i_in, opt=(av[i_sc] - av[i_in]) * 1e5))
    log(f"fold {k}: trees picked on scored fold {its[i_sc]} (AUC {av[i_sc]:.6f}), on inner 10% {its[i_in]} "
        f"(scored AUC {av[i_in]:.6f}) -> optimism {ES[-1]['opt']:+.1f}e-5")
del X

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw=dict(width_ratios=[1.7, 1]))
FC = [C[0], C[6], C[3]]
for e, col in zip(ES, FC):
    rel = (e["av"] - e["av"][e["i_sc"]]) * 1e5
    ax.plot(e["its"], rel, color=col, lw=1.6, label=f"fold {e['fold']}")
    ax.plot(e["its"][e["i_sc"]], 0, "o", ms=8, color=col, mec=SURF, mew=1.5, zorder=5)
    ax.plot(e["its"][e["i_in"]], rel[e["i_in"]], "D", ms=7, mfc=SURF, mec=col, mew=1.8, zorder=5)
low = min((e["av"][e["i_in"]] - e["av"][e["i_sc"]]) * 1e5 for e in ES)
ax.set_ylim(low * 1.6 - 5, 8); ax.set_xlim(100, ES_ROUNDS); ax.yaxis.set_major_formatter(E5)
ax.set_xlabel("number of trees"); ax.set_ylabel("scored-fold AUC minus its maximum (x 1e-5)")
ax.legend(handles=[Line2D([], [], color=c, lw=2, label=f"fold {e['fold']}") for e, c in zip(ES, FC)] +
          [Line2D([], [], marker="o", ls="", color=INK2, label="trees chosen on the scored fold"),
           Line2D([], [], marker="D", ls="", mfc=SURF, mec=INK2, label="trees chosen on a held-out 10%")], loc="lower right", fontsize=8.5)
panel_title(ax, "The top of a noisy curve is always the highest point", "scored-fold AUC along training, 4-leaf LightGBM, learning rate 0.1")
opts = [e["opt"] for e in ES]
ax2.bar(range(len(ES)), opts, color=PRIV, width=0.55)
for i, v in enumerate(opts): ax2.text(i, v + 0.3, f"+{v:.1f}", ha="center", va="bottom", fontsize=10, fontweight="bold", color=INK)
ax2.set_xticks(range(len(ES))); ax2.set_xticklabels([f"fold {e['fold']}" for e in ES]); ax2.yaxis.set_major_formatter(E5)
ax2.set_ylabel("optimism of the reported fold AUC (x 1e-5)"); ax2.set_ylim(0, max(opts) * 1.3 + 1)
panel_title(ax2, f"Free optimism: {np.mean(opts):+.1f}e-5 per fold", "scored-fold pick minus held-out pick")
headline(fig, f"Early stopping on the scored fold reports {np.mean(opts):.0f} points more than an honest choice of trees",
         "Same model, two ways to choose the number of trees: on the fold that is scored, or on a held-out 10% of the training part")
plt.show()

The held-out 10% is itself a noisy judge, so its pick lands away from the scored-fold peak: that is the point. The
scored-fold pick hides this noise and reports the peak; any choice made without the scored rows pays for it.
A fast learner (learning rate 0.1) is the bad case: its curve is jagged, so the maximum of the scored fold sits well
above what a choice made elsewhere achieves. With the slower learning rate most public tree members use (0.03) a
separate measurement of the same comparison gave about **+2 to +3e-5 per member**. Per member that is small next to
fold noise, but it is **systematic**: a stack of a hundred members
that all stopped on their scored fold inherits it, and stack weights then favour the members whose OOF is most
optimistic. Choosing the number of trees on a held-out slice of the training part
costs nothing and removes it.

In [ ]:
# 5.2  Selecting on the reported folds: greedy hill climbing (Caruana) over the OOF bank's 20 core members.
# Non-nested: choose the blend on all OOF rows, report its CV on the same rows.  Nested: choose on 4 folds, score the 5th.
names = list(BANK); M = np.stack([BANK[n] for n in names], 1).astype(np.float32)
HC_STEPS = 10
def hill_climb(rows):
    """Greedy forward selection with replacement on `rows`; returns the list of picks (indices into names)."""
    yr = y[rows]; Mr = M[rows]
    first = int(np.argmax([auc_sort(yr, Mr[:, j]) for j in range(M.shape[1])]))
    picks, s = [first], Mr[:, first].astype(np.float64).copy()
    for _ in range(HC_STEPS - 1):
        sc = [auc_sort(yr, s + Mr[:, j]) for j in range(M.shape[1])]
        j = int(np.argmax(sc)); picks.append(j); s += Mr[:, j]
    return picks
def blend(picks, upto): return M[:, picks[:upto]].mean(1)
full_picks = hill_climb(np.arange(N))
HC_NON = [cv(blend(full_picks, t)) for t in range(1, HC_STEPS + 1)]
HC_NEST = np.zeros((5, HC_STEPS))
for k, b in enumerate(FOLDS):
    p = hill_climb(np.setdiff1d(np.arange(N), b))
    for t in range(1, HC_STEPS + 1): HC_NEST[k, t - 1] = roc_auc_score(y[b], blend(p, t)[b])
    log(f"  hill climbing, outer fold {k}: picks {[names[j] for j in p[:4]]} ...")
HC_NEST_CV = HC_NEST.mean(0)
BEST_SINGLE = max(cv(M[:, j]) for j in range(M.shape[1]))
log(f"hill climbing: reported (non-nested) CV {HC_NON[-1]:.6f}, nested CV {HC_NEST_CV[-1]:.6f}, gap {(HC_NON[-1] - HC_NEST_CV[-1]) * 1e5:+.1f}e-5")

In [ ]:
# 5.3  A test inside one fold: does adding megayak's LM (already a member of the wide stack) to W help?
# Outer test: weights fitted on the other four folds, scored on fold k.  Inner test: a 5-fold CV inside fold k's rows only.
Z = np.c_[W, O["LM"]]
INOUT = []
for k, b in enumerate(FOLDS):
    rest = np.setdiff1d(np.arange(N), b)
    lr_ = LogisticRegression(C=1.0, max_iter=500).fit(Z[rest], y[rest])
    outer = roc_auc_score(y[b], Z[b] @ lr_.coef_[0]) - roc_auc_score(y[b], W[b])
    ins = []
    for it, ib in StratifiedKFold(5, shuffle=True, random_state=0).split(b, y[b]):
        l2 = LogisticRegression(C=1.0, max_iter=500).fit(Z[b[it]], y[b[it]])
        ins.append(roc_auc_score(y[b[ib]], Z[b[ib]] @ l2.coef_[0]) - roc_auc_score(y[b[ib]], W[b[ib]]))
    INOUT.append((k, np.mean(ins) * 1e5, outer * 1e5))
    print(f"  fold {k}: inner test {np.mean(ins) * 1e5:+6.1f}e-5   outer test {outer * 1e5:+6.1f}e-5")

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw=dict(width_ratios=[1.25, 1]))
t = np.arange(1, HC_STEPS + 1)
ax.plot(t, (np.array(HC_NON) - BEST_SINGLE) * 1e5, color=PRIV, marker="o", ms=6, mec=SURF, label="reported: chosen and scored on the same OOF")
ax.plot(t, (HC_NEST_CV - BEST_SINGLE) * 1e5, color=CV, marker="o", ms=6, mec=SURF, label="nested: chosen on 4 folds, scored on the 5th")
gap = (HC_NON[-1] - HC_NEST_CV[-1]) * 1e5
ax.annotate(f"gap {gap:+.1f}e-5", xy=(HC_STEPS, (HC_NON[-1] - BEST_SINGLE) * 1e5), xytext=(-8, 14), textcoords="offset points",
            ha="right", fontsize=10, fontweight="bold", color=INK)
ax.set_xlabel("hill-climbing step (members in the blend, with repeats)"); ax.set_ylabel("CV gain over the best single member (x 1e-5)")
ax.yaxis.set_major_formatter(E5); ax.set_xticks(t); ax.legend(loc="lower right", fontsize=9)
panel_title(ax, f"Selection on the scored rows reads {gap:.0f} points high", "greedy blend of the OOF bank's 20 core members")
ks = [r[0] for r in INOUT]
for k, i_, o_ in INOUT:
    ax2.plot([i_, o_], [k, k], color=RULE, lw=3, zorder=1)
ax2.scatter([r[1] for r in INOUT], ks, s=70, color=PRIV, zorder=3, label="inner CV inside the fold", edgecolor=SURF, lw=1.5)
ax2.scatter([r[2] for r in INOUT], ks, s=70, color=CV, zorder=3, label="outer test on the fold", edgecolor=SURF, lw=1.5)
zero_line(ax2, "x"); ax2.set_yticks(ks); ax2.set_yticklabels([f"fold {k}" for k in ks]); ax2.invert_yaxis()
ax2.xaxis.set_major_formatter(E5); ax2.set_xlabel("gain from adding a member already in the stack (x 1e-5)")
ax2.legend(loc="lower right", fontsize=9)
n_dis = sum(np.sign(i_) != np.sign(o_) for _, i_, o_ in INOUT)
panel_title(ax2, f"Inner and outer verdicts disagree on {n_dis} of 5 folds", "logistic re-weighting of the wide stack + megayak LM")
headline(fig, "Where you select and where you score must be different rows",
         "Left: a blend picked on the OOF it is scored on. Right: a decision taken inside one fold versus across folds.")
plt.show()

<div style="background: #fff4ec; border-left: 5px solid #eb6834; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>Rules of thumb for an honest CV.</b> (1) Stop trees on a held-out slice of the training part, never on the scored
fold. (2) Every choice a pipeline makes (members, weights, C, gates, transforms) is made inside the training part of
each outer fold: that is <i>nested</i> CV. (3) Decide across all outer folds, not inside one: a fold-local test sees
that fold's own calibration and noise. (4) A stack fitted once on the full OOF and then reported on the same OOF reads
high; fit it again fold by fold to get the number you can compare.
</div>

<a id="s6"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">6 &middot; What did not work</div>

Negative results are rarely published and save everyone time. First, public probes that the ladder authors reported
as losses (all credit to them for publishing these):

| probe on a ladder file | public change | reported by |
|---|---|---|
| + 0.12 of the public target-first GPT-2 (already inside every ladder file) | -14 | goodpjw2008 (v1 probes) |
| + 5% / + 2% TabPFN-3.5 | -1 / -1 | Areeb Ali (v18 and probes) |
| 10% mix with another public stack | -1 | Areeb Ali |
| + 10% / + 20% of a GPT-2 + GLM fusion file | 0 / 0 | Areeb Ali |
| more of both LMs (+0.04, +0.03) | -2 | Areeb Ali |
| an extra LM step on daoviet's file, other LM kept at 0.12 | 0 | goodpjw2008 |

Areeb Ali's summary: *"new model families (TabPFN, GBDT files, other stacks) lose a point"*. On nested CV, adding more
language-model weight to a wide stack also loses (-2.8e-5, up on only 2 of 5 folds in our check), even though public
ladders gained points from LM steps.

Second, "smarter" ways to combine a bank of public OOF predictions, each compared on **nested CV over 5 outer folds**
with every setting fixed in advance, against simply keeping a strong logistic stack:

| method | nested CV vs. the plain stack | folds better |
|---|---|---|
| greedy hill climbing (Caruana), bank that already contains the stack | +0.0 (it picks the stack alone) | 0/5 |
| logistic regression on all ~80 deduplicated members, C by inner CV | -27.7e-5 | 0/5 |
| weights proportional to pairwise AUC | -40.6e-5 | 0/5 |
| two experts gated by a data segment | -30.9e-5 | 0/5 |
| stack + a small segment-gate correction | -1.9e-5 | 0/5 |
| non-linear meta-model (4-leaf LightGBM on members, member disagreement, raw columns) | -39e-5 to -56e-5 | 0&ndash;1/5 |

<div style="background: #eefaf4; border-left: 5px solid #1baf7a; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>Pattern.</b> Once a linear stack holds the strong members, cleverer combiners mostly fit noise: they lose on
every outer fold by far more than any ladder step gains. The one kind of change that kept gaining on nested CV in the
public record is <b>new information</b> (another seed or another view of a strong model), not new weights.
</div>

<a id="s7"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">7 &middot; What to expect on private, and how to pick two final files</div>

Put the pieces together for the current public top file. We replay its checkable steps on the nested OOF, in the
order they were built, and set them beside the public points each one earned.

In [ ]:
D_a3 = 0.12 * (O["LM"] - W)
D_352 = 0.09 * (O["LM"] - W) + 0.09 * (O["L7F"] - W) + 0.05 * (O["GPD"] - W) - 0.05 * (O["MLP"] - W) - 0.025 * (O["GB"] - W)
c_a3, c_352 = W + D_a3, W + D_352
c_354 = L(c_352) + 0.06 * (O["G26"] - O["MS"])
CHAIN = [("Strike 9 -> Strike 10", 10, None, None), ("+ 0.6 (S10 - S9)", 2, None, None),
         ("+ 0.12 megayak LM", 1, (cv(c_a3) - CV_W) * 1e5, folds_up(c_a3, W)),
         ("-> 0.95352 file\n(4 steps)", 9, (cv(c_352) - cv(c_a3)) * 1e5, folds_up(c_352, c_a3)),
         ("+ GPT-2 seed swap", 2, (cv(c_354) - cv(c_352)) * 1e5, folds_up(c_354, c_352))]
CHECK_PUB = sum(c[1] for c in CHAIN if c[2] is not None); CHECK_CV = sum(c[2] for c in CHAIN if c[2] is not None)
for lab, p, c, u in CHAIN:
    print(f"  {lab.replace(chr(10), ' '):28s} public {p:+3d}   nested CV " + ("  no OOF" if c is None else f"{c:+5.1f}  ({u}/5 folds up)"))
print(f"checkable steps: public {CHECK_PUB:+d} points, nested CV {CHECK_CV:+.1f} points")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.0))
x = np.arange(len(CHAIN))
ax.bar(x - 0.2, [c[1] for c in CHAIN], width=0.38, color=PUB, label="public LB points earned")
for i, (lab, p, c, u) in enumerate(CHAIN):
    ax.text(i - 0.2, p + 0.3, f"+{p}", ha="center", va="bottom", fontsize=10, fontweight="bold", color=INK)
    if c is None:
        ax.bar(i + 0.2, 4, width=0.38, color="none", edgecolor=MUTED, hatch="///", lw=0)
        ax.text(i + 0.2, 4.3, "no OOF:\nuntestable", ha="center", va="bottom", fontsize=8.5, color=INK2)
    else:
        ax.bar(i + 0.2, c, width=0.38, color=CV if (c > 0 and u >= 4) else PRIV)
        ax.text(i + 0.2, c + (0.3 if c >= 0 else -0.3), f"{c:+.1f}\n({u}/5)", ha="center", va="bottom" if c >= 0 else "top", fontsize=9, color=INK)
zero_line(ax); ax.set_xticks(x); ax.set_xticklabels([c[0] for c in CHAIN], fontsize=9.5); ax.yaxis.set_major_formatter(E5)
ax.set_ylabel("points (x 1e-5)"); ax.grid(axis="x", visible=False)
ax.set_ylim(min(-8, min(c[2] for c in CHAIN if c[2] is not None) - 3), 14)
ax.legend(handles=[Patch(color=PUB, label="public LB points earned"), Patch(color=CV, label="nested CV: confirmed"),
                   Patch(color=PRIV, label="nested CV: not confirmed"), Patch(facecolor="none", edgecolor=MUTED, hatch="///", label="no OOF published")],
          loc="upper right", fontsize=9)
headline(fig, f"Of the +24 public points from Strike 9 to the top file, the checkable +{CHECK_PUB} are worth {CHECK_CV:+.1f} on nested CV",
         "Each step of the 0.95354 ladder file replayed on the nested OOF of the public wide LM stack (moves applied in the same order)")
plt.show()

**What this means for the private board** (an estimate, not a forecast):

* The absolute private score of every file can sit a few *thousandths* away from its public score, in the same
  direction for everyone. Nobody can know that direction; it does not change the ranking.
* The **ranking** inside the top band depends on gaps of a few points, and each gap carries a public noise of about
  one ladder-rung SD. Expect the top of the private board to reshuffle by that much.
* Steps that nested CV did not confirm have an expected private value near their CV value, not their public value.
  Steps without any OOF (the base files and the extrapolation) are a genuine unknown: they may hold, or not.
* The two-seed rung is the exception that proves the method: it won on both boards and has a mechanism.

<div style="background: #eefaf4; border-left: 5px solid #1baf7a; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>Picking two final files &mdash; general advice.</b><br>
<b>1. One CV-backed file.</b> Every choice in it (members, weights, early stopping) made inside the training folds; its
nested CV is the number you trust. Its public score may sit below the ladders; that is fine.<br>
<b>2. One different hypothesis.</b> If you believe a public ladder's untestable base carries real signal, that is a
legitimate second pick &mdash; but make the second file genuinely <i>different</i> from the first (lower rank
correlation), so the two picks hedge each other.<br>
<b>3. Never two near-copies.</b> Two files at rank correlation 0.9999 land within a few points of each other on
private: the second slot is wasted.<br>
<b>4. Do not let the public board choose between close files.</b> If two candidates differ by less than about two
noise SDs (use <code>public_gap_noise()</code> above), the public board cannot tell them apart; decide by CV or by
reasoning.
</div>

<a id="s8"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">8 &middot; New in v4: the board, 12 &rarr; 14 November</div>

In two days the band moved up by about three hundred-thousandths and two new public notebooks took the top of the
Code list. The leaderboard snapshot below is from 14 Nov, 01:03 UTC (scores and submission counts only).

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 5.0), gridspec_kw=dict(width_ratios=[1.25, 1]))
rk12, rk14 = np.arange(1, len(LB_TOP) + 1), np.arange(1, len(LB14_TOP) + 1)
ax.plot(rk12, LB_TOP, color=MUTED, lw=2, label="12 Nov, 12:11 UTC")
ax.plot(rk14, LB14_TOP, color=PUB, lw=2.2, label="14 Nov, 01:03 UTC")
for v in (354, 370, 388): ax.axhline(v, color=RULE, lw=1, zorder=0)
ax.text(330, 520, "thin lines, top to bottom: public notebook files\n0.95388 Bold, Then Bolder  ·  0.95370 rank-1 q17 file\n0.95354 two-seed ladders (the v3 top)",
        fontsize=8.5, color=INK2, ha="right", va="bottom")
for i, lab in [(0, "#1"), (1, "#2")]:
    ax.annotate(f"{lab}  0.95{LB14_TOP[i]:03d}", (rk14[i], LB14_TOP[i]), xytext=(14, -4), textcoords="offset points", fontsize=9,
                fontweight="bold", color=INK, va="center")
ax.set_xscale("log"); ax.set_xlim(0.8, 340); ax.set_ylim(300, 800)
ax.set_xticks([1, 2, 5, 10, 20, 50, 100, 300]); ax.set_xticklabels(["1", "2", "5", "10", "20", "50", "100", "300"])
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"0.95{int(v):03d}"))
ax.set_xlabel("leaderboard rank"); ax.set_ylabel("public LB score"); ax.legend(loc="upper right", fontsize=9)
panel_title(ax, "The band rose; two scores sit far above it", "public score at every rank, both snapshots")
# panel B: where #1 sits, in noise SDs of the gap between two DIFFERENT strong files
gaps = (LB14_TOP[:12] - BAND14) / FAR_SD
cols = [PRIV if g > 2 else PUB for g in gaps]
ax2.bar(np.arange(1, 13), gaps, color=cols, width=0.68, zorder=3)
ax2.axhspan(-2, 2, color=BAND, zorder=0, lw=0)
ax2.text(12.4, 1.7, "within 2 noise SDs of #5", ha="right", va="top", fontsize=8.5, color=INK2)
for i, g in enumerate(gaps[:3]):
    ax2.text(i + 1, g + 0.15, f"{g:.1f}", ha="center", va="bottom", fontsize=9.5, fontweight="bold", color=INK)
zero_line(ax2); ax2.set_xticks(np.arange(1, 13)); ax2.set_xlabel("leaderboard rank (14 Nov)")
ax2.set_ylabel("score minus #5, in noise SDs"); ax2.grid(axis="x", visible=False); ax2.set_ylim(-1.0, gaps[0] * 1.15)
panel_title(ax2, f"#1 is {gaps[0]:.1f} noise SDs above #5", f"1 SD = {FAR_SD:.0f} points: public gap noise of two different strong files (rho {FAR_RHO:.3f})")
headline(fig, f"In two days the band rose ~{LB14_TOP[49] - LB_TOP[49]} points at #50; #1 at 0.95{LB14_TOP[0]} is far outside any public-noise range",
         f"Leaderboard snapshots 12 Nov ({LB_TEAMS:,} teams) and 14 Nov ({LB14_TEAMS:,} teams); noise from random 20/80 splits of the public wide-stack OOF")
plt.show()

**How to read panel B.** The noise SD used here is the *largest* one in this notebook: the public gap between two
genuinely different strong files (the wide LM stack and the best of mathsonchain's single models, rank correlation
about 0.987), measured on random 20/80 splits. Gaps between ladder files are ten times less noisy. Even on this
generous scale, #1 is many SDs above the band, so public noise does not explain it; nothing else about that entry is
known from public data, and this notebook makes no claim about it. #2 is also well above the band. From #5 down, the
board is again a dense band in which neighbours are well inside each other's noise.

<a id="s9"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">9 &middot; New in v4: anatomy of the two new ladders</div>

Both new top notebooks publish every probe with its public score and an OOF check of each mix, which is what makes
the replay below possible. Thank you to **goodpjw2008** and **Areeb Ali** again, and to **Aman Atar** for the bundle
both are built on.

* **goodpjw2008, [LM Ladder | Public and CV Up | LB 0.95386](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-public-and-cv-up-lb-0-95386).**
  Starts from Aman Atar's public 0.95370 file (with 0.06 of one LM moved to the other) and moves it towards 87 public
  members. The weights maximise *OOF change + mu x a forecast of the public change*, where the forecast is a model
  fitted on 44 of the author's own scored files (not shipped). Three settings were submitted: careful 0.95376,
  balanced 0.95380, **bold 0.95386**.
* **Areeb Ali, [Bold, Then Bolder | LB 0.95388](https://www.kaggle.com/code/areebbb/s6e11-bold-then-bolder-lb-0-95388).**
  Rebuilds the three mixes byte for byte and probes eight steps beyond bold on the board. Keeps
  **bold + 0.5 (bold - balanced) = 0.95388**.

```
U0    = rank(AT + 0.06 (LM7 - LM))                         # AT = the public 0.95370 file (Aman Atar)
mix   = (1 - sum_k w_k) logit(U0) + sum_k w_k member_k      # 87 members, logit ranks
w     = w_bold + 0.5 (w_bold - w_balanced)                  # 0.95388
```

Every mix is linear in its weights, so every probe of both notebooks can be rebuilt on the test rows (for its noise)
and on the OOF rows (for its CV) from the same public inputs. The base file has no OOF of its own; its OOF stand-in is
goodpjw2008's replay of it from the 16 bundle models and the wide LM stack (R&sup2; shown in the log above).

In [ ]:
STEPS9 = [   # (label, file, previous file, public change, author)
    ("0.95354 -> rank-1 q17 file", "0.95370 file", "0.95354 file", 16, "goodpjw2008 v3 / Aman Atar"),
    ("careful mix", "careful", "0.95370 file", 6, "goodpjw2008"),
    ("balanced mix", "balanced", "0.95370 file", 10, "goodpjw2008"),
    ("bold mix (kept)", "bold", "0.95370 file", 16, "goodpjw2008"),
    ("1.25 x bold", "1.25 x bold", "bold", 1, "Areeb Ali"),
    ("1.5 x bold", "1.5 x bold", "bold", 1, "Areeb Ali"),
    ("bold + 0.5 (bold - careful)", "bold + 0.5 (bold - careful)", "bold", 1, "Areeb Ali"),
    ("bold + 0.5 (bold - balanced) (kept)", "bold + 0.5 (bold - balanced)", "bold", 2, "Areeb Ali"),
    ("bold + 1.0 (bold - balanced)", "bold + 1.0 (bold - balanced)", "bold", 1, "Areeb Ali"),
    ("bold + 1.5 (bold - balanced)", "bold + 1.5 (bold - balanced)", "bold", 1, "Areeb Ali"),
    ("bold + 2.0 (bold - balanced)", "bold + 2.0 (bold - balanced)", "bold", -1, "Areeb Ali"),
    ("1.25 x (bold + 1.0 (bold - balanced))", "1.25 x (bold + 1.0 (bold - balanced))", "bold", 0, "Areeb Ali"),
]
NESTED_AUTH = {"careful": 22.9, "balanced": 14.3, "bold": 19.5}       # the authors' nested CV (weights refitted on 4 folds)
rows9 = []
for lab, f, prev, pub, who in STEPS9:
    sd = sig(rho_te(f, prev)) if TE.get(prev) is not None else np.nan
    oof_prev = "ref" if prev == "0.95370 file" else prev
    if f in PO and oof_prev in FA:
        c, d = chg(f, oof_prev); up = int((d > 0).sum())
    else: c, d, up = np.nan, None, None
    rows9.append(dict(step=lab, author=who, pub=pub, sd=sd, z=pub / sd if sd else np.nan, cv=c, up=up, nested=NESTED_AUTH.get(f, np.nan), folds=d))
T9 = pd.DataFrame(rows9)
show9 = pd.DataFrame({"step (vs the file it was built on)": T9.step, "by": T9.author, "public change": T9.pub.map(lambda v: f"{v:+d}"),
                      "public noise SD": T9.sd.map(lambda v: f"{v:.1f}"), "public / noise": T9.z.map(lambda v: f"{v:+.1f}"),
                      "OOF change (in-sample)": T9.cv.map(lambda v: "no OOF" if np.isnan(v) else f"{v:+.1f}"),
                      "folds up": T9.up.map(lambda v: "-" if v is None or v != v else f"{int(v)}/5"),
                      "authors' nested CV": T9.nested.map(lambda v: "-" if np.isnan(v) else f"{v:+.1f}")})
print("All changes in points of 1e-5. Noise SD = K x sqrt(1 - rho) on the two TEST files (the one-line rule of section 2).")
show9.style.hide(axis="index") if hasattr(show9.style, "hide") else show9

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7.2))
yy = np.arange(len(T9))[::-1]
for yi, r in zip(yy, T9.itertuples()):
    ax.add_patch(FancyBboxPatch((-2 * r.sd, yi - 0.36), 4 * r.sd, 0.72, boxstyle="round,pad=0,rounding_size=0.1", facecolor=BAND, edgecolor="none", zorder=0))
    ax.add_patch(FancyBboxPatch((-r.sd, yi - 0.36), 2 * r.sd, 0.72, boxstyle="round,pad=0,rounding_size=0.1", facecolor=RULE, edgecolor="none", zorder=0))
    ax.plot(r.pub, yi + 0.13, "o", ms=9, color=PUB, mec=SURF, mew=1.4, zorder=4)
    if not np.isnan(r.cv):
        ax.plot(r.cv, yi - 0.13, "D", ms=7.5, color=CV if (r.up >= 4 and r.cv > 0) else PRIV if r.cv < 0 else MUTED, mec=SURF, mew=1.2, zorder=4)
        ax.text(max(r.cv, r.pub, 2 * r.sd) + 1.2, yi, f"OOF {r.cv:+.1f} ({int(r.up)}/5 folds up)", va="center", fontsize=8.5, color=INK2)
    else:
        ax.text(max(r.pub, 2 * r.sd) + 1.2, yi, "no OOF: untestable", va="center", fontsize=8.5, color=MUTED)
    if not np.isnan(r.nested): ax.plot(r.nested, yi - 0.13, "D", ms=7.5, mfc=SURF, mec=CVTXT, mew=1.6, zorder=5)
ax.axhline(len(T9) - 4.5, color=RULE, lw=1); ax.axhline(len(T9) - 1.5, color=RULE, lw=1)
ax.text(-38, len(T9) - 1.0, "base switch", fontsize=8.5, color=MUTED, va="center")
ax.text(-38, len(T9) - 3.0, "goodpjw2008:\nthree mixes vs\nthe 0.95370 file", fontsize=8.5, color=MUTED, va="center")
ax.text(-38, 3.5, "Areeb Ali:\neight probes\nvs the bold mix", fontsize=8.5, color=MUTED, va="center")
ax.set_yticks(yy); ax.set_yticklabels(T9.step, fontsize=9); ax.tick_params(axis="y", length=0)
zero_line(ax, "x"); ax.xaxis.set_major_formatter(E5); ax.set_xlim(-40, 62); ax.grid(axis="y", visible=False)
ax.set_xlabel("change vs the file the step was built on (x 1e-5)")
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=PUB, ms=8, label="public LB change"),
                   Line2D([], [], marker="D", ls="", color=CV, ms=7, label="OOF change, up on >= 4 of 5 folds"),
                   Line2D([], [], marker="D", ls="", color=MUTED, ms=7, label="OOF change, fewer folds up"),
                   Line2D([], [], marker="D", ls="", color=PRIV, ms=7, label="OOF change < 0"),
                   Line2D([], [], marker="D", ls="", mfc=SURF, mec=CVTXT, mew=1.6, ms=7, label="authors' nested CV"),
                   Patch(color=RULE, label="1 SD public noise"), Patch(color=BAND, label="2 SD")],
          loc="upper center", bbox_to_anchor=(0.45, -0.11), ncol=4, fontsize=8.5)
_k = T9[T9.step == "bold + 0.5 (bold - balanced) (kept)"].iloc[0]
headline(fig, f"The new ladders: the mixes are big moves with OOF behind them; the kept last step is {_k.pub / _k.sd:.1f} noise SD",
         "Every public probe of both notebooks, rebuilt from public files: public change, its noise, and the OOF change of the same move")
plt.show()

**What the anatomy says.**

* **The base switch (+16) is a different file, not a step.** The 0.95370 file is Aman Atar's; its rank correlation with
  the 0.95354 ladder file is far from 1, so its public noise is larger, and it has no OOF. It is the one link in the new
  ladders that nobody can check.
* **The three mixes are real moves with OOF behind them.** They are large moves (public noise SD of about 8 to 13
  points), and their OOF gains (+19 to +33 in-sample, +14 to +23 in the authors' nested CV) are larger than that noise
  and positive on all five folds. Moves of this size are something the public board *can* see, roughly.
* **Areeb Ali's eight probes are inside the noise.** Their public changes (-1 to +2) are all within one noise SD of
  zero, and the kept probe is the best of eight such numbers. On OOF the probes along `bold - balanced` gain a few
  points on 4 or 5 folds; scaling the whole mix gains less and less consistently, and stepping along `bold - careful`
  loses.

<a id="s10"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">10 &middot; New in v4: "public and CV up" &mdash; checked fold by fold</div>

goodpjw2008's claim is that the mixes raise **both** the public score and CV. Two checks on our replay:

1. **Fold by fold.** A CV-confirmed step should gain on at least 4 of 5 folds, by a margin larger than the
   fold-to-fold spread. The mix weights saw all five folds, so these are in-sample numbers (the authors' nested
   numbers, with weights refitted on four folds, are shown beside them).
2. **Nested choice of the step size.** Areeb Ali chose *how far* to go along `bold - balanced` on the public board.
   Here the four other folds choose it, and the fifth scores it, for each fold in turn.

In [ ]:
# (2) nested step size along d = bold - balanced: t chosen on four folds, scored on the fifth, gain vs the bold mix.
TGRID = np.round(np.arange(0, 2.01, 0.25), 2)
_d = PO["bold"] - MO["balanced"]
TG = np.array([[auc_sort(y[b], (PO["bold"] + t * _d)[b]) for b in FOLDS] for t in TGRID]) * 1e5
TG = TG - TG[0]                                          # gain over bold, (t, fold)
T_OUT, T_PICK = np.zeros(5), np.zeros(5)
for f in range(5):
    j = TG[:, [g for g in range(5) if g != f]].mean(1).argmax(); T_PICK[f], T_OUT[f] = TGRID[j], TG[j, f]
print("nested step size along (bold - balanced): picks", T_PICK, " outer gains", np.round(T_OUT, 1), f" mean {T_OUT.mean():+.1f}, up {(T_OUT > 0).sum()}/5")
print("the kept half step (t = 0.5): fold gains", np.round(TG[2], 1))
CHK10 = [("careful", "ref"), ("balanced", "ref"), ("bold", "ref"), ("bold + 0.5 (bold - balanced)", "bold"),
         ("1.5 x bold", "bold"), ("bold + 1.5 (bold - balanced)", "bold"), ("388 + 5% GP + DVAE", "bold + 0.5 (bold - balanced)"),
         ("388 + 5% shallow tree", "bold + 0.5 (bold - balanced)")]
LAB10 = {"careful": "careful mix vs 0.95370", "balanced": "balanced mix vs 0.95370", "bold": "bold mix vs 0.95370",
         "bold + 0.5 (bold - balanced)": "0.95388 step vs bold", "1.5 x bold": "1.5 x bold vs bold",
         "bold + 1.5 (bold - balanced)": "bold + 1.5 (bold - bal.) vs bold",
         "388 + 5% GP + DVAE": "+5% GP + DVAE on 0.95388", "388 + 5% shallow tree": "+5% shallow tree on 0.95388"}
V10 = []
for a, b in CHK10:
    c, d = chg(a, b); V10.append((LAB10[a], c, d, (d > 0).sum(), d.std(ddof=1)))
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 5.8), gridspec_kw=dict(width_ratios=[1.45, 1]))
yy = np.arange(len(V10))[::-1]
for yi, (lab, c, d, up, sd) in zip(yy, V10):
    hold = up >= 4 and abs(c) > sd / np.sqrt(5) * 2
    col = CV if (hold and c > 0) else PRIV if c < 0 and up <= 1 else MUTED
    ax.plot([d.min(), d.max()], [yi, yi], color=RULE, lw=2.5, solid_capstyle="round", zorder=1)
    ax.scatter(d, np.full(5, yi), s=26, color=col, alpha=0.8, edgecolor="none", zorder=3)
    ax.plot(c, yi, "|", ms=16, mew=2.5, color=INK, zorder=4)
    ax.text(1.01, yi, f"{c:+.1f}  {up}/5", va="center", ha="left", fontsize=9, color=CVTXT if col == CV else INK2, fontweight="bold" if col == CV else "normal",
            transform=ax.get_yaxis_transform(), clip_on=False)
ax.set_yticks(yy); ax.set_yticklabels([v[0] for v in V10], fontsize=9.5); ax.tick_params(axis="y", length=0)
zero_line(ax, "x"); ax.xaxis.set_major_formatter(E5); ax.set_xlim(-14, max(v[2].max() for v in V10) + 3); ax.grid(axis="y", visible=False)
ax.set_xlabel("OOF AUC change per fold (x 1e-5); black tick = change on all rows")
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=CV, label="holds: >= 4/5 folds, clear of the fold spread"),
                   Line2D([], [], marker="o", ls="", color=MUTED, label="fold noise: small or mixed"),
                   Line2D([], [], marker="o", ls="", color=PRIV, label="down on >= 4 folds")], loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, fontsize=8.5)
panel_title(ax, "The mixes hold on every fold; small steps are mostly fold noise", "five dots = five folds")
for f in range(5): ax2.plot(TGRID, TG[:, f], color=C[[0, 6, 3, 4, 5][f]], lw=1.3, alpha=0.75, label=f"fold {f}")
ax2.plot(TGRID, TG.mean(1), color=INK, lw=2.4, label="mean of the folds")
ax2.axvline(0.5, color=PUB, lw=1.2, ls=(0, (3, 2))); ax2.text(0.55, ax2.get_ylim()[1] * 0.55, "public pick\nt = 0.5", color=PUB, fontsize=8.5, va="top")
zero_line(ax2); ax2.yaxis.set_major_formatter(E5)
ax2.set_xlabel("t in  bold + t (bold - balanced)"); ax2.set_ylabel("OOF change vs bold, per fold (x 1e-5)"); ax2.legend(loc="upper left", fontsize=8, ncol=2)
panel_title(ax2, f"Nested step size: {T_OUT.mean():+.1f}, {(T_OUT > 0).sum()}/5 folds up", "inner folds pick t = " + ", ".join(f"{t:g}" for t in T_PICK))
headline(fig, "\"Public and CV up\" holds for the three mixes on every fold; the kept half step is small, and one fold carries about half of it",
         "OOF rows of the public files; the mix weights saw all five folds (in-sample), the step size on the right is chosen nested")
plt.show()

<a id="s9v"></a>
### What each new notebook offers, and what its public tuning is likely to cost on private

For every step that was **kept because it won on the board**, the public number is the best of several. The model of
section 4, now run on the real probes of each author: on each random 20/80 split of the OOF rows (real public and
private sizes), the probe with the best public gap is kept, and its private gap is read. The difference is the
**give-back**. A step's expected private change is its public change minus the give-back, with a 90% interval from the
same draws. Steps chosen without the board, or scored once, get the plain noise interval.

In [ ]:
def interval_k1(g, sd):
    s = np.sqrt(sd ** 2 + (sd * np.sqrt(NPUB / (NT - NPUB))) ** 2); return g, g - 1.645 * s, g + 1.645 * s
VERD = []
gp = GIVE["goodpjw2008"]; ga = GIVE["Areeb Ali"]
r_base = T9.iloc[0]
VERD.append(("Aman Atar / goodpjw2008", "0.95354 -> rank-1 q17 file", 16, *interval_k1(16, r_base.sd), np.nan, np.nan, "probes unknown: plain noise", -1))
for lab, gv, fam, oof_n, ref_n in [("goodpjw2008", 16, gp, "bold", "ref"), ("Areeb Ali", 2, ga, "bold + 0.5 (bold - balanced)", "bold")]:
    e = gv - fam["back"]
    VERD.append((lab, "bold mix (best of 5)" if lab == "goodpjw2008" else "half step (best of 8)", gv, e.mean(), np.percentile(e, 5), np.percentile(e, 95),
                 chg(oof_n, ref_n)[0], NESTED_AUTH.get(oof_n, np.nan), f"best of {fam['K']}: give-back {fam['back'].mean():+.1f}", int((chg(oof_n, ref_n)[1] > 0).sum())))
for lab, n_, pub in [("Sameer Shaikh (Part 3)", "388 + 5% GP + DVAE", -3), ("Sameer Shaikh (Part 3)", "388 + 5% shallow tree", -1)]:
    rr = np.corrcoef(rankdata(PO[n_]), rankdata(MO["388"]))[0, 1]
    VERD.append((lab, n_.replace("388", "0.95388"), pub, *interval_k1(pub, sig(rr)), chg(n_, "bold + 0.5 (bold - balanced)")[0], np.nan, "not kept: plain noise",
                 int((chg(n_, "bold + 0.5 (bold - balanced)")[1] > 0).sum())))
VT = pd.DataFrame(VERD, columns=["notebook", "step", "public", "exp_private", "lo", "hi", "oof", "nested", "model", "up"])
def verdict(r):
    if np.isnan(r.oof): return "unknown: no OOF"
    if r.oof >= 10 and r.up >= 4: return "real gain on OOF;\npublic alone is weak evidence"
    if r.oof > 0 and r.up >= 4: return "tiny gain on OOF;\npublic number is noise"
    if r.oof <= 0: return "no gain on OOF"
    return "fold noise"
VT["verdict"] = VT.apply(verdict, axis=1)
showv = pd.DataFrame({"notebook": VT.notebook, "step": VT.step, "public change": VT.public.map(lambda v: f"{v:+d}"),
                      "expected private change (90% interval)": [f"{m:+.1f}  [{l:+.1f}, {h:+.1f}]" for m, l, h in zip(VT.exp_private, VT.lo, VT.hi)],
                      "how": VT.model, "OOF change (in-sample)": [("-" if np.isnan(v) else f"{v:+.1f} ({u}/5 up)") for v, u in zip(VT.oof, VT.up)],
                      "authors' nested": VT.nested.map(lambda v: "-" if np.isnan(v) else f"{v:+.1f}"), "verdict": VT.verdict.str.replace("\n", " ")})
showv.style.hide(axis="index") if hasattr(showv.style, "hide") else showv

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.3))
yy = np.arange(len(VT))[::-1]
for yi, r in zip(yy, VT.itertuples()):
    ax.plot([r.lo, r.hi], [yi - 0.12, yi - 0.12], color=PRIV, lw=3, solid_capstyle="round", alpha=0.45, zorder=2)
    ax.plot(r.exp_private, yi - 0.12, "o", ms=9, color=PRIV, mec=SURF, mew=1.4, zorder=4)
    ax.plot(r.public, yi + 0.14, "o", ms=8, color=PUB, mec=SURF, mew=1.4, zorder=4)
    if not np.isnan(r.oof): ax.plot(r.oof, yi + 0.14, "D", ms=7, color=CV, mec=SURF, mew=1.2, zorder=4)
    if not np.isnan(r.nested): ax.plot(r.nested, yi + 0.14, "D", ms=7, mfc=SURF, mec=CVTXT, mew=1.6, zorder=5)
    ax.text(1.015, yi, r.verdict, va="center", ha="left", fontsize=8.8, color=INK, transform=ax.get_yaxis_transform(), clip_on=False,
            fontweight="bold" if r.verdict.startswith("real") else "normal")
ax.set_yticks(yy); ax.set_yticklabels([f"{a}\n{b}" for a, b in zip(VT.notebook, VT.step)], fontsize=8.8); ax.tick_params(axis="y", length=0)
zero_line(ax, "x"); ax.xaxis.set_major_formatter(E5); ax.set_xlim(-32, 52); ax.grid(axis="y", visible=False)
ax.set_xlabel("change vs the file the step was built on (x 1e-5)")
ax.legend(handles=[Line2D([], [], marker="o", ls="", color=PUB, label="public change"),
                   Line2D([], [], marker="o", ls="-", color=PRIV, label="expected private change, 90% interval"),
                   Line2D([], [], marker="D", ls="", color=CV, label="OOF change (in-sample)"),
                   Line2D([], [], marker="D", ls="", mfc=SURF, mec=CVTXT, mew=1.6, label="authors' nested CV")],
          loc="upper center", bbox_to_anchor=(0.42, -0.16), ncol=4, fontsize=8.5)
headline(fig, "What each new notebook's public steps are likely to keep on private",
         "Expected private change = public change minus the give-back of picking the best public probe, simulated on the OOF rows (real 20/80 sizes)")
plt.show()

**Reading the verdicts.**

* **goodpjw2008's mixes offer real signal.** The bold mix gains on the OOF rows in all five folds and the authors' own
  nested CV keeps +19.5 of it. On the board alone, +16 is only about 1.3 noise SDs for a move this size, and keeping
  the best of five probes costs about six points in expectation, so the interval is wide; the OOF evidence is what
  makes it convincing. This is the strongest new public evidence since v3.
* **Areeb Ali's half step is small but real on OOF; the board cannot see it.** On OOF it adds about four points, up on
  5 of 5 folds (in-sample, one fold carrying about half of it). A nested choice of the step size also gains, on 4 of 5 folds,
  but prefers a longer step (t around 1.5), as the authors found on CV. The public +2 is the best of eight numbers with
  a noise of about three points, so the public score itself is no evidence: corrected for the selection, its expected
  private value is about -3 (the interval is wide because several of the eight probes are large moves, and any of them
  can win a random split). Take the file for the bold mix it contains, not for the last point.
* **The base switch has no OOF.** Aman Atar's 0.95370 file is a strong public file whose private value nobody can
  check from public data.
* **The two Part-3 probes on top of 0.95388** (Sameer Shaikh's team, both lost on the board) lose or barely move on
  OOF too: 5% more GP + DVAE loses on 4 of 5 folds, 5% more shallow tree gains under two points. In a file that already
  holds a member at a fitted weight, adding more of it does not help.

The other new notebooks are not public-tuned ladders, so there is no give-back to estimate:

| notebook | what it offers | public tuning |
|---|---|---|
| Sameer Shaikh, [stack of every public OOF bank](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-wide-pooled-stack-every-bank-found) | one logistic stack over every public OOF bank on the shared split, scored by nested CV (author: 0.95255) | none: a CV-backed file |
| Sameer Shaikh, [Public Submission Fusion (No OOF)](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-public-submission-fusion-no-oof) | an honest log of file fusions, including probes on top of 0.95388 that lost | every number is public-only, and the author says so |
| mathsonchain, [What Worked, What Didn't](https://www.kaggle.com/code/mathsonchain/s6e11-fraud-what-worked-what-didn-t) | a careful progress report and [17 single models with OOF](https://www.kaggle.com/datasets/mathsonchain/s6e11-fraud-oof-test-17-single-models) on the shared split (used in section 12) | none |
| denpugovkin, [Scale-Aware Digit GLM](https://www.kaggle.com/code/denpugovkin/s6e11-scale-aware-digit-glm) | a paired, pre-declared experiment on a single GLM | none |
| daoviet, [On the journey to your first top 100](https://www.kaggle.com/code/daoviet/s6e11-on-the-journey-to-your-first-top-100) | a beginner-friendly walk through the public ladder and the GP + DVAE member | reproduces public steps |

<a id="s11"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">11 &middot; New in v4: error analysis &mdash; which rows do the top files fight over?</div>

The OOF rows are the stand-in test set: 699,960 rows with labels, predicted out of fold. Four files are compared on
them: the honest **wide LM stack** (nested OOF, goodpjw2008), the **0.95370 replay**, the **bold mix** and the
**0.95388 file** (both built on the replay exactly as on test). Every number below is an exact AUC: a step is applied
to a **subset of rows only**, and the AUC change of the whole file is read. The subsets are analysis tools; a label
based subset (frauds only) could of course never be used to predict.

In [ ]:
for k in EA: print(f"  {k:16s} OOF AUC {EA_AUC[k]:.6f}   frauds caught in the top 1% / 2% / 5% of rows: " + " / ".join(f"{c:.1%}" for c in CAPTURE[k]))
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.9), gridspec_kw=dict(width_ratios=[1, 1.1]))
STC = {STEPS_EA[0]: PUB, STEPS_EA[1]: C[6], STEPS_EA[2]: PRIV}
STL = {STEPS_EA[0]: "0.95370 replay -> bold mix", STEPS_EA[1]: "bold mix -> 0.95388 file", STEPS_EA[2]: "wide LM stack -> 0.95388 file"}
for st, dd in DISAG.items():
    full = dd["gain"][-1]
    ax.plot(np.array(QGRID) * 100, dd["gain"], "o-", color=STC[st], ms=5, lw=2, label=f"{STL[st]}  (all rows {full:+.1f})")
ax.set_xscale("log"); ax.set_xticks([0.5, 1, 2, 5, 10, 20, 50, 100]); ax.set_xticklabels(["0.5", "1", "2", "5", "10", "20", "50", "100"])
zero_line(ax); ax.yaxis.set_major_formatter(E5); ax.legend(loc="upper left", fontsize=8.5)
ax.set_xlabel("% of rows that take the new file's rank (most-moved rows first)"); ax.set_ylabel("AUC change of the whole file (x 1e-5)")
panel_title(ax, "The change is spread over many rows", "rows ranked by how far the step moves them; the most-moved are mostly legit rows")
seg = SEG_TAB[(SEG_TAB.segmentation == "score band (rank in the base file)")]
bands = ["bottom 80%", "80-95%", "95-99%", "top 1%"]
x = np.arange(len(bands)); wdt = 0.26
for i, st in enumerate(STEPS_EA):
    v = [seg[(seg.step == f"{st[0]} -> {st[1]}") & (seg.segment == b)].gain.iloc[0] for b in bands]
    ax2.bar(x + (i - 1) * wdt, v, width=wdt * 0.92, color=STC[st], label=STL[st], zorder=3)
    for xi, vi in zip(x + (i - 1) * wdt, v):
        ax2.text(xi, vi + (0.6 if vi >= 0 else -0.6), f"{vi:+.0f}", ha="center", va="bottom" if vi >= 0 else "top", fontsize=8, color=INK2)
fr = [seg[(seg.step == f"{STEPS_EA[0][0]} -> {STEPS_EA[0][1]}") & (seg.segment == b)].frauds.iloc[0] for b in bands]
ax2.set_xticks(x); ax2.set_xticklabels([f"{b}\n({f:.0%} of frauds)" for b, f in zip(bands, fr)], fontsize=9)
zero_line(ax2); ax2.yaxis.set_major_formatter(E5); ax2.grid(axis="x", visible=False)
ax2.set_ylabel("AUC change if only this band moves (x 1e-5)")
lo_, hi_ = ax2.get_ylim(); ax2.set_ylim(lo_ - 2, hi_ + 3)
panel_title(ax2, "The gains come from rows below the top 5%", "band = rank of the row in the step's starting file")
headline(fig, "The mixes gain by lifting frauds that the starting file ranks low, not by reordering its top 5%",
         "Exact OOF AUC changes when a step is applied to a subset of rows only; the 0.95388 file's OOF uses the replayed base")
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 4.6), sharey=False)
for ax, st in zip(axs, STEPS_EA):
    sh = SHIFT[st]; xc = np.arange(len(EDGES) - 1)
    ax.plot(xc, sh[:, 0], "o-", color=PRIV, ms=5, lw=1.8, label="frauds")
    ax.plot(xc, sh[:, 1], "o-", color=MUTED, ms=4, lw=1.5, label="legit rows")
    ax.fill_between(xc, sh[:, 1], sh[:, 0], where=sh[:, 0] >= sh[:, 1], color=CV, alpha=0.12, lw=0, interpolate=True)
    ax.fill_between(xc, sh[:, 1], sh[:, 0], where=sh[:, 0] < sh[:, 1], color=PRIV, alpha=0.12, lw=0, interpolate=True)
    zero_line(ax)
    ax.set_xticks(xc); ax.set_xticklabels([f"{EDGES[i] * 100:g}" for i in range(len(EDGES) - 1)], fontsize=8, rotation=0)
    ax.set_xlabel("rank band of the starting file (lower edge, %)")
    panel_title(ax, STL[st], "mean rank change, x 1e-3 of all rows")
axs[0].set_ylabel("mean change of rank fraction (x 1e-3)"); axs[0].legend(loc="upper right", fontsize=8.5)
headline(fig, "Where a step lifts frauds above legit rows (green) and where it drops them (orange)",
         "Mean rank shift of frauds and legit rows by the starting file's rank band; the AUC also depends on which legit rows each fraud passes")
plt.show()

In [ ]:
# Segment table: AUC change (x 1e-5) when only the segment moves, its share of rows and frauds, how far its rows move
# (mean |rank change| x 1e-3), and how much more the step lifts its frauds than its legit rows (x 1e-3).
tab11 = SEG_TAB.copy()
tab11["rows"] = tab11.rows.map(lambda v: f"{v:.1%}"); tab11["frauds"] = tab11.frauds.map(lambda v: f"{v:.1%}")
tab11["gain"] = tab11.gain.map(lambda v: f"{v:+.1f}"); tab11["moved"] = tab11.moved.map(lambda v: f"{v:.2f}")
tab11["lift"] = tab11.lift.map(lambda v: "-" if v != v else f"{v:+.2f}")
tab11 = tab11.rename(columns={"gain": "AUC change if only it moves", "moved": "mean |rank change|", "lift": "frauds minus legit shift"})
tab11.style.hide(axis="index") if hasattr(tab11.style, "hide") else tab11

**What the error analysis says** (all on the OOF stand-in, so a guide rather than a forecast):

* **The top 1% is settled.** All four files catch the same share of frauds in their top 1%, 2% and 5% of rows (to a
  tenth of a percent). The steps between them hardly touch the order of the top rows.
* **The mixes gain lower down.** The step from the 0.95370 replay to the bold mix gains almost all of its AUC in rows
  the replay ranks *below* its top 5%: about a quarter of all frauds sit there, and the mix lifts them clearly more
  than the legit rows around them (left rank-shift panel). Its members carry signal about frauds that the base file
  ranks too low, which is also why the gain shows on every fold.
* **The change is spread over many rows.** The 1% of rows that move furthest are almost all legit rows in the middle
  of the score; they carry only a small part of the gain, and half of the rows must take the new ranks before most of
  it appears. These are not a few lucky rows.
* **The honest stack is still ahead on OOF.** Moving from the wide LM stack to the 0.95388 file's OOF counterpart
  loses, and again the loss sits below the top 1%, mostly in the 80&ndash;99% bands. The replay
  is an approximation (about 1% of the 0.95370 file has no OOF counterpart, and goodpjw2008 notes that the replay is
  if anything too hard on the file), so this is a measured hint, not a verdict on the file.
* **Calibration of the order.** A step that helps does not shift every fraud up; it shifts frauds up *relative to
  legit rows of the same band*. The rank-shift panels show that gap band by band. They are a description, not the
  whole story: in the third panel the 0.95388 file lifts low-ranked frauds more than legit rows *on average* and still
  loses AUC against the wide stack, because the AUC counts how many legit rows each fraud ends up above, row by row,
  not the average move.

<a id="s12"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">12 &middot; New in v4: can one honest step improve the best public file?</div>

This notebook's `submission.csv` starts from the best public file, **Areeb Ali's 0.95388**, rebuilt from public inputs
above. On top of it we allow **at most two steps**, and only steps that pass a rule fixed before looking at any result:

1. **Candidates.** A small step `B + w (M - B)` in logit-rank space towards (or away from) one public member `M` that
   has honest OOF on the shared five folds and is **not** one of the 87 members already in the mix: the 17 single
   models of mathsonchain's dataset, goodpjw2008's wide stack without LMs and his EBM bag. `w` is one of
   &plusmn;0.01, &plusmn;0.02, &plusmn;0.03, &plusmn;0.05. That is 152 candidate steps.
2. **Nested choice.** For each of the five folds in turn, the other four folds pick the best candidate (member and
   weight); the held-out fold scores it.
3. **Gate.** Accept a step only if it gains on **at least 4 of 5** held-out folds and on average. The public board is
   never consulted.

If no step passes, the output is the plain rebuild of the 0.95388 file.

In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw=dict(width_ratios=[1.35, 1]))
rd = EXT[0]
cols = [CV if g > 0 else PRIV for g in rd["outer"]]
ax.bar(np.arange(5), rd["outer"], color=cols, width=0.6, zorder=3)
for f, (g, (m, w)) in enumerate(zip(rd["outer"], rd["picks"])):
    ax.text(f, g + (0.4 if g >= 0 else -0.4), f"{g:+.1f}", ha="center", va="bottom" if g >= 0 else "top", fontsize=10, fontweight="bold", color=INK)
ylo = min(rd["outer"].min(), 0) - 4.5; yhi = max(rd["outer"].max(), 0) + 3
ax.set_ylim(ylo, yhi)
for f, (m, w) in enumerate(rd["picks"]):
    ax.text(f, ylo + 0.4, f"{m.split(' (')[0]}\nw = {w:+.2f}", ha="center", va="bottom", fontsize=7.8, color=INK2)
zero_line(ax); ax.set_xticks(range(5)); ax.set_xticklabels([f"held-out fold {f}" for f in range(5)], fontsize=9)
ax.yaxis.set_major_formatter(E5); ax.set_ylabel("AUC change on the held-out fold (x 1e-5)"); ax.grid(axis="x", visible=False)
panel_title(ax, f"Step 1: up on {(rd['outer'] > 0).sum()} of 5 held-out folds, mean {rd['outer'].mean():+.1f}: "
                f"{'PASSES' if rd['passed'] else 'does not pass'} the gate", "the step each set of four folds picked (bottom) and how it did on the fifth")
vals = [rd["final_in"], rd["outer"].mean()]
ax2.bar([0, 1], vals, color=[MUTED, CV if vals[1] > 0 else PRIV], width=0.55, zorder=3)
for i, v in enumerate(vals): ax2.text(i, v + (0.25 if v >= 0 else -0.25), f"{v:+.1f}", ha="center", va="bottom" if v >= 0 else "top", fontsize=11, fontweight="bold", color=INK)
zero_line(ax2); ax2.set_xticks([0, 1]); ax2.set_xticklabels(["best of 152 steps,\nchosen and scored\non the same five folds", "the same search,\nnested (chosen on four,\nscored on the fifth)"], fontsize=9)
ax2.yaxis.set_major_formatter(E5); ax2.set_ylabel("OOF change (x 1e-5)"); ax2.grid(axis="x", visible=False)
lo2 = min(vals) - 2; ax2.set_ylim(min(lo2, -2), max(vals) + 2)
panel_title(ax2, "Selection inflates CV too", "the winner's curse of section 4, inside cross-validation")
headline(fig, "No candidate step passes the nested gate: the output stays the plain rebuild of the 0.95388 file" if N_STEPS == 0
         else f"{N_STEPS} step(s) pass the nested gate and are applied on top of the 0.95388 file",
         "152 small steps towards or away from 19 public members with honest OOF, chosen inside the folds; the public board is never used")
plt.show()

In [ ]:
# Shown: the output of this notebook.
print(f"rebuild of the best public file (0.95388, Areeb Ali) identical to the published one: {REBUILD_OK}")
print(f"extension steps that passed the nested gate: {N_STEPS}")
for i, rd_ in enumerate(EXT):
    res_ = "passed: %s %+.2f" % rd_["final"] if rd_["passed"] else "not passed"
    print(f"  round {i + 1}: held-out gains {np.round(rd_['outer'], 1)}, up on {(rd_['outer'] > 0).sum()}/5, mean {rd_['outer'].mean():+.1f} -> {res_}")
print(f"submission.csv SHA-256: {SUB_SHA}")
print("submission.csv = the rebuilt 0.95388 file" if N_STEPS == 0 else "submission.csv = the 0.95388 file plus the step(s) above")
pd.read_csv(WORK / "submission.csv").head()

<div style="background: #eefaf4; border-left: 5px solid #1baf7a; padding: 10px 14px; border-radius: 6px; margin: 6px 0;">
<b>Why "no step" is a useful result.</b> Searched on all five folds at once, the best of 152 candidate steps looks like a
gain of a few points: exactly the kind of number a ladder rung is made of. Chosen honestly, inside the folds, the same
search does not gain on enough held-out folds. The members tested here are good models, but the 0.95388 file already
contains the information they carry. The next real gain is a member with errors that nobody in the mix shares,
checked on OOF first &mdash; the conclusion the ladder authors themselves reach.
</div>

<a id="s13"></a>
<div style="background: #0b1f4b; color: #ffffff; padding: 9px 16px; border-radius: 8px; font-size: 20px; font-weight: 700;">13 &middot; Full credits and licences</div>

**Notebooks whose published numbers are quoted** (public scores, probe tables, formulas):

* goodpjw2008 &mdash; [S6E11 LM Ladder | Two GPT-2 Seeds | LB 0.95354](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-two-gpt-2-seeds-lb-0-95354),
  [LM Ladder Part 1](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-part-1-own-lm-step-lb-0-95346),
  [Part 12 | Wide Stack + 2nd LM](https://www.kaggle.com/code/goodpjw2008/s6e11-part-12-wide-stack-2nd-lm-lb-0-95298)
  (the wide LM stack and its nested OOF), and the `find()` helper and logit-rank convention used here.
* Areeb Ali &mdash; [S6E11 | Three Ladders, Two Seeds | LB 0.95354](https://www.kaggle.com/code/areebbb/s6e11-three-ladders-two-seeds-lb-0-95354)
  and [Three Ladders, Two Steps](https://www.kaggle.com/code/areebbb/s6e11-three-ladders-two-steps-lb-0-95347): version history, probe table, CV-evidence table.
* amanatar &mdash; [S6E11 | Podium Apex | Public LB 0.95340](https://www.kaggle.com/code/amanatar/s6e11-podium-apex-public-lb-0-95340) (Strike 10).
* Moran Z &mdash; [S6E11 | Grandmaster Ladder | LB 0.95330](https://www.kaggle.com/code/moranzavdi/s6e11-grandmaster-ladder-lb-0-95330) (Strike 9).
* masayakawamata &mdash; [GPT-2 Target-First Classifier](https://www.kaggle.com/code/masayakawamata/s6e11-gpt-2-target-first-classifier-cv-0-9500); with Matapanino, the target-first GPT-2.
* megayak &mdash; [per-value digit TE shallow-tree stack](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack) and the target-first LM.
* samanyu1808 &mdash; the LM-step idea on these files (as credited by goodpjw2008 and Areeb Ali).
* **New in v4.** goodpjw2008 &mdash; [S6E11 LM Ladder | Public and CV Up | LB 0.95386](https://www.kaggle.com/code/goodpjw2008/s6e11-lm-ladder-public-and-cv-up-lb-0-95386)
  (the replay of the 0.95370 file, the careful / balanced / bold weights and their nested CV, every probe);
  Areeb Ali &mdash; [S6E11 | Bold, Then Bolder | LB 0.95388](https://www.kaggle.com/code/areebbb/s6e11-bold-then-bolder-lb-0-95388)
  (the 0.95388 weights, its SHA-256 and eight probes); Aman Atar &mdash; the rank-1 q17 bundle (the 0.95370 file and 16 public
  models, whose authors include megayak, masayakawamata, slimypunk03, Sadam Torres, yekenot, alexchilton, daoviet,
  Sameer Shaikh and goodpjw2008); Sameer Shaikh &mdash; [Public Submission Fusion (No OOF)](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-public-submission-fusion-no-oof)
  (the probes on top of 0.95388) and [stack of every public OOF bank](https://www.kaggle.com/code/sameershaikh27/s6e11-fraud-wide-pooled-stack-every-bank-found);
  mathsonchain &mdash; [What Worked, What Didn't](https://www.kaggle.com/code/mathsonchain/s6e11-fraud-what-worked-what-didn-t);
  denpugovkin &mdash; [Scale-Aware Digit GLM](https://www.kaggle.com/code/denpugovkin/s6e11-scale-aware-digit-glm);
  daoviet &mdash; [On the journey to your first top 100](https://www.kaggle.com/code/daoviet/s6e11-on-the-journey-to-your-first-top-100).
* Chris Deotte &mdash; [S6E10 1st place write-up](https://www.kaggle.com/competitions/playground-series-s6e10/writeups/1st-place-tfms-and-llms-with-tabular-data), the target-first language-model idea.

**Datasets attached to this notebook** (each file checked by SHA-256):

| dataset | files used | licence |
|---|---|---|
| [goodpjw2008/s6e11-fraud-series-artifacts](https://www.kaggle.com/datasets/goodpjw2008/s6e11-fraud-series-artifacts) | `wide_lm_stack_nested.npy`, `wide_lm_stack_test.npy`; v4: `wide_stack_*`, `ebm_b4_*` | CC0-1.0 |
| [goodpjw2008/s6e11-oof-bank](https://www.kaggle.com/datasets/goodpjw2008/s6e11-oof-bank) | all 70 members (OOF + test), `manifest.csv` | CC0-1.0 |
| [goodpjw2008/s6e11-compact-lm-seed7](https://www.kaggle.com/datasets/goodpjw2008/s6e11-compact-lm-seed7) | seed-7 LM, OOF + test; the frozen copy of the target-first LM's test file | CC0-1.0 |
| [goodpjw2008/s6e11-target-first-gpt2-seed-2026](https://www.kaggle.com/datasets/goodpjw2008/s6e11-target-first-gpt2-seed-2026) | GPT-2 seed 2026, OOF + test | CC0-1.0 |
| [megayak/s6e11-target-first-lm](https://www.kaggle.com/datasets/megayak/s6e11-target-first-lm) | target-first LM, OOF + test | CC0-1.0 |
| [masayakawamata/s6e11-gpt2-target-first-oof](https://www.kaggle.com/datasets/masayakawamata/s6e11-gpt2-target-first-oof) | target-first GPT-2, OOF + test | CC BY 4.0 |
| [daoviet/s6e11-gp-dvae-orthogonal-apex](https://www.kaggle.com/datasets/daoviet/s6e11-gp-dvae-orthogonal-apex) | GP + DVAE, OOF + test (until v3; the author removed these files on 13 Nov, and v4 reads the same predictions, rank-identical, from Aman Atar's bundle) | CC0-1.0 |
| [amanatar/s6e11-strike10-podium-apex-095340](https://www.kaggle.com/datasets/amanatar/s6e11-strike10-podium-apex-095340) | `submission_095340.csv` | CC0-1.0 |
| [amanatar/s6e11-rank1-q17-meta-bundle](https://www.kaggle.com/datasets/amanatar/s6e11-rank1-q17-meta-bundle) | `rank1_q17_streams.npz` (v4) | CC0-1.0 |
| [mathsonchain/s6e11-fraud-oof-test-17-single-models](https://www.kaggle.com/datasets/mathsonchain/s6e11-fraud-oof-test-17-single-models) | 17 single models, OOF + test (v4) | CC BY 4.0 |
| [moranzavdi/s6e11-strike9-lb-095330](https://www.kaggle.com/datasets/moranzavdi/s6e11-strike9-lb-095330) | `submission_095330.csv` | CC0-1.0 |

The competition data belongs to Kaggle's Playground Series S6E11 (competition rules apply). The leaderboard snapshots
(scores and submission counts only, 2026-11-12 12:11 UTC and 2026-11-14 01:03 UTC) are from the public leaderboard page.
This notebook's code is released under Apache-2.0. Its `submission.csv` is the public 0.95388 file of Areeb Ali, rebuilt
from the inputs above (no extension step passed the nested gate of section 12); the credit for it goes to Areeb Ali,
goodpjw2008, Aman Atar and the authors of the members it contains.

Thank you again to everyone above. If this notebook helped you, please upvote the source notebooks too, starting with
goodpjw2008, Areeb Ali and Aman Atar.

In [ ]:
log("done")